<a href="https://colab.research.google.com/github/Leewonseol/Korean-history-visualization-v2-HTML/blob/main/STRICT_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STRICT_ML — 2단계 Q11 만족도 분석 파이프라인

- **Step 1**: 1~3점 vs 4~5점 → **PRIM** (탈출 방지) + **5-Fold CV**
- **Step 2**: 4점 vs 5점 → **RuleFit** (팬심 공략) + **5-Fold CV**

Group C (Q11=5 vs 4 역방향 RuleFit)는 주석 처리.
X_rule: 특수결측 복원·변수 필터만 (MICE 미사용).

In [10]:
# ── Colab 환경 설정 (로컬에서 실행 시 자동 스킵) ──────────────────────────────
import sys, os

# IN_COLAB 정의는 가장 먼저
IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # 레포 클론 (rulefit_elasticnet.py 등 커스텀 모듈 포함)
    if not os.path.exists('PORT_ONE'):
        os.system('git clone https://github.com/leewonseol/port_one.git PORT_ONE')

    # 'PORT_ONE' 디렉토리의 절대 경로를 sys.path에 추가 (가장 확실한 방법)
    port_one_abs_path = os.path.abspath('PORT_ONE')
    if port_one_abs_path not in sys.path:
        sys.path.insert(0, port_one_abs_path)

    # 작업 디렉토리 변경 (다른 파일 경로 처리에 일관성을 제공)
    if os.getcwd() != port_one_abs_path:
        os.chdir('PORT_ONE')

    # _DATA_CSV를 여기에 정의하여 모든 곳에서 접근 가능하도록 함
    # Colab 환경에서는 chdir이 수행되므로, 현재 작업 디렉토리 기준의 절대 경로로 만듭니다.
    _DATA_CSV = Path(os.getcwd()) / "2024_외래관광객조사_Data_수정_no_col1.csv"
    print(f'Colab 환경: PORT_ONE 레포 로드 완료 → {os.getcwd()} (데이터 파일: {_DATA_CSV})')
else:
    print('로컬 환경: 레포 클론 불필요')
    # 로컬 환경에서는 현재 스크립트 파일이 있는 디렉토리를 기준으로 합니다.
    _DATA_CSV = Path("2024_외래관광객조사_Data_수정_no_col1.csv") # 로컬 경로에 파일이 있다고 가정

# 모든 필요한 임포트 구문은 여기에 (환경 설정 후)
import warnings
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Set, Tuple
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold

# rulefit_elasticnet 모듈 임포트
# 이제 sys.path에 PORT_ONE 경로가 있으므로 임포트 가능
from rulefit_elasticnet import (
    run_rulefit_elasticnet_for_target,
    run_multi_booster_rulefit,
    merge_multi_booster_results,
    summarize_rulefit_en_results,
    summarize_linear_terms,
)

warnings.filterwarnings("ignore", category=RuntimeWarning, message="overflow encountered")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="invalid value encountered")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.utils.validation")

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# ═════════════════════════════════════════════════════════════════════════════
# Config
# ═════════════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    data_csv: Path = Path("data.csv")
    output_dir: Path = Path("./output")
    target: str = "Q11"
    demographics: List[str] = field(default_factory=lambda: [
        "D_MON", "D_NAT", "D_SEX", "D_AGE", "D_MOK", "D_NUM", "D_GUB", "D_BUN"
    ])
    random_seed: int = 2026
    prim_min_support: int = 10
    # PRIM/RuleFit 전용 X_rule (특수결측 복원·필터만; MICE 미사용)
    prim_rule_missing_rate_threshold: float = 0.5  # 0.3 등으로 실험 시 변경
    prim_rule_num_missing_code: float = -69.0
    prim_rule_cat_missing_code: float = -77.0
    prim_rule_ambiguous_name_substrings: Tuple[str, ...] = (
        "_dk", "dk", "_DK", "DK",
    )
    # 지출/금액류: 음수가 나올 수 있어 비정상 음수 판정에서 제외할 이름 prefix
    prim_rule_allow_negative_prefixes: Tuple[str, ...] = (
        "MQ7", "Q7", "KWON", "RQ7", "WQ9", "Q9",
    )
    # 다중응답(MQ*)에서 결측 의미가 불명확할 수 있어, 결측률이 이 값 이상이면 제거 (None이면 규칙 비활성)
    prim_rule_multi_response_missing_drop_threshold: Optional[float] = 0.35
    # 이진 변수에서 한 범주 비중이 이 값 이상이면 '응답 여부만'에 가깝다고 보고 제거
    prim_rule_quasi_constant_binary_mass: float = 0.995
    # 코드북 prefix로 잡히지 않는 변수: 연속형(비율) vs 구간범주(C*) 명시
    prim_rule_explicit_numerical_features: Tuple[str, ...] = (
        "Q2_3b2e", "Q2_3b3e", "Q2_3b2d", "Q2_3b3c", "Q2_3b3d", "Q2_3b3b",
        "Q2_3b2c", "Q2_3b2b", "Q2_3b3a", "Q2_3b2a",
        "학교숙박60", "민박숙박60", "민박숙박", "학교숙박",
        "모텔숙박60", "모텔숙박", "게스숙박60", "게스숙박",
        "친척숙박60", "친척숙박",
        "한국국외1인대체",
        "숙박비1인대체", "국제한국1인대체", "국제국외1인대체", "국제수상1인대체",
        "가이드1인대체", "음식점1인대체", "식음료1인대체",
        "한국한국1인대체", "한국수상1인대체", "한국철도1인대체", "한국도로1인대체",
        "대여서1인대체", "유류비1인대체", "문화서1인대체", "오락및1인대체",
        "쇼핑비1인대체", "데이터1인대체", "치료및1인대체", "미용서1인대체", "기타비1인대체",
    )
    prim_rule_explicit_categorical_features: Tuple[str, ...] = (
        "C한국국외1인대체",
        "C국제한국1인대체", "C국제국외1인대체", "C국제수상1인대체",
        "C가이드1인대체", "C음식점1인대체",
        "C한국한국1인대체", "C한국수상1인대체", "C한국철도1인대체", "C한국도로1인대체",
        "C대여서1인대체", "C유류비1인대체", "C문화서1인대체", "C오락및1인대체",
        "C데이터1인대체", "C치료및1인대체", "C미용서1인대체", "C기타비1인대체",
    )
    test_size: float = 0.2
    catboost_threads: int = 4
    shap_n_sample: int = 2000
    verbose: bool = True

    # ============================================================================
    # 2024 외래관광객조사 코드북 기준 변수 척도 분류 (Scale Classification)
    # ============================================================================
    # 명목척도(Nominal) + 서열척도(Ordinal) → 범주형(categorical)
    # 비율척도(Ratio) → 수치형(numeric)
    #
    # prefix 기반 매칭: 변수명이 해당 prefix로 시작하면 해당 척도로 분류
    # exact name 매칭: prefix로 분류 불가한 변수는 정확한 이름으로 매칭
    # ============================================================================
    nominal_prefixes: Tuple[str, ...] = (
        # 기본 식별/인구통계: D_SEX, D_NAT, D_NAT2, D_MON
        "D_SEX", "D_NAT", "D_MON",
        # 여행 형태/방한 목적: TYP, D_GUB, Q1, D_MOK, Q6
        "TYP", "D_GUB", "D_MOK",
        # 동반자 유형: Q7A, Q7a_dk~Q7a8
        "Q7A", "Q7a",
        # 관광 행태/선택 요인 (다중/우선순위): Q1_1a, Q3_1a, Q3_2a
        "Q1_1a", "Q3_1a", "Q3_2a",
        # 참여활동/만족활동: Q8a, Q8_1a
        "Q8a", "Q8_1a",
        # 정보수집/예약: Q4a, Q4_1a, Q4_2a, Q5_1a
        "Q4a", "Q4_1a", "Q4_2a", "Q5_1a",
        # 이동동선 (방문국가/지역): Q2a, Q2_1a, Q2_3a, ZQ2
        "Q2a", "Q2_1a", "Q2_3a", "ZQ2",
        # 방문지역/가장좋았던곳: Q9_1_, Q9_2a
        "Q9_1_", "Q9_2a",
        # 방문권역: KWON
        "KWON",
        # 숙박시설 시도별/유형: WQ9_5a, Q9_5A
        "WQ9_5a", "Q9_5A",
        # 쇼핑: Q10_2a, Q10_3a
        "Q10_2a", "Q10_3a",
    )
    nominal_exact: Tuple[str, ...] = (
        "pnid", "Q1", "Q6", "Q7A",
    )
    ordinal_prefixes: Tuple[str, ...] = (
        # 만족도/행동의도 (리커트): Q12a
        "Q12a",
        # 방문횟수 범주화: RVIT, XRVIT, D_NUM
        "RVIT", "XRVIT",
        # 체류기간 범주화: R1일HAP
        "R1일HAP",
        # 동반자수 범주화: RQ7
        "RQ7",
        # 지출경비 범주화 (금액 구간): C총액, C여행사, C단기투어, RDAY, C교통, C숙박, C식음료, C쇼핑
        "C총액", "C여행사", "C단기투어", "RDAY", "C교통", "C숙박", "C식음료", "C쇼핑",
    )
    ordinal_exact: Tuple[str, ...] = (
        "Q11", "Q13", "Q14", "Q5", "D_AGE", "D_BUN", "D_NUM",
        "Q6_1R",
    )
    ratio_prefixes: Tuple[str, ...] = (
        # 방문/체류 실측값 (일/박, 횟수): MVIT, M박HAP, M일HAP
        "MVIT", "M박HAP", "M일HAP",
        # 시도별 숙박/체재: 서울박, 서울일, 부산박, 부산일, ...
        "서울박", "서울일", "부산박", "부산일", "대구박", "대구일",
        "인천박", "인천일", "광주박", "광주일", "대전박", "대전일",
        "울산박", "울산일", "세종박", "세종일",
        "경기박", "경기일", "강원박", "강원일",
        "충북박", "충북일", "충남박", "충남일",
        "전북박", "전북일", "전남박", "전남일",
        "경북박", "경북일", "경남박", "경남일",
        "제주박", "제주일",
        # 숙박시설별 숙박일수: 호텔숙박, 콘도숙박, 게스트하우스숙박, ...
        "호텔숙박", "콘도숙박", "게스트하우스숙박", "기타숙박",
        # 동반자수 실측: MQ7
        "MQ7",
        # 지출경비 절대금액: 총액1인, MDAY
        "총액1인", "MDAY",
        # 세부항목 지출금액 (한글): 여행사1인, 교통1인, 숙박1인, 식음료1인, 쇼핑1인
        "여행사1인", "단기투어상품1인",
        # 쇼핑비중: MQ10_2b
        "MQ10_2b",
        # 가중치: weight
        "weight",
    )
    ratio_exact: Tuple[str, ...] = (
        "Q6_1",
    )
    # ============================================================================
    # 저장 정책 플래그 (SAVE_LEVEL)
    # ============================================================================
    # "minimal": 최소한의 최종 요약 파일만 저장
    #   - model_metrics_validation.csv, model_metrics_full.csv
    #   - metric_best_models.json, metric_best_models.csv
    #   - validation 기준 지표별 최고 모델에 대한 PRIM/SHAP/CPI 결과만 저장
    # "full": 모든 중간 결과 포함 (디버깅용)
    save_level: str = "minimal"  # "minimal" 또는 "full"

    # 저장 허용 파일 패턴 (save_level="minimal"일 때만 적용)
    # True: 저장 허용, False: 저장 비활성화
    save_threshold_grid: bool = False  # threshold_grid_val_*.csv
    save_threshold_candidates: bool = False  # threshold_candidates_val_*.csv
    save_threshold_alpha_scenarios: bool = False  # *_threshold_alpha_scenarios.csv
    save_threshold_strategies: bool = False  # threshold_strategies_summary.csv
    save_threshold_metrics_full: bool = False  # threshold_metrics_full.csv
    save_model_rankings: bool = False  # model_ranking_*.csv
    save_all_models_prim: bool = False  # 모든 모델의 PRIM 결과
    save_all_models_prim_subgroups: bool = False  # 모든 모델의 PRIM subgroups 통합

    def __post_init__(self):
        self.data_csv = Path(self.data_csv)
        self.output_dir = Path(self.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)


Colab 환경: PORT_ONE 레포 로드 완료 → /content/PORT_ONE (데이터 파일: /content/PORT_ONE/2024_외래관광객조사_Data_수정_no_col1.csv)


In [ ]:
if IN_COLAB:
    from google.colab import drive

    # Check if drive is already mounted and unmount if it is, to avoid 'Mountpoint must not already contain files' error
    if os.path.exists('/content/drive') and os.path.ismount('/content/drive'):
        print('Google Drive is already mounted. Attempting to unmount first.')
        try:
            drive.flush_and_unmount()
            print('Google Drive unmounted successfully.')
        except Exception as e:
            print(f'Error unmounting Google Drive: {e}. Attempting to mount anyway.')

    # Attempt to mount Google Drive
    try:
        drive.mount('/content/drive', force_remount=True)
        print('Google Drive mounted at /content/drive')
    except ValueError as e:
        print(f'ValueError during drive mount: {e}. This might happen if the mountpoint is not empty.')
        print('Please check /content/drive and ensure it is empty or unmount it manually if needed.')
    except Exception as e:
        print(f'An unexpected error occurred during drive mount: {e}')
else:
    print('Not in Colab, Google Drive mounting skipped.')

ValueError during drive mount: Mountpoint must not already contain files. This might happen if the mountpoint is not empty.
Please check /content/drive and ensure it is empty or unmount it manually if needed.


## 1. 임포트 및 환경 설정

In [ ]:
#!/usr/bin/env python3
"""STRICT_ML — 2단계 Q11 만족도 분석 파이프라인.

Step 1: 1~3점 vs 4~5점 → PRIM ("탈출 방지": 최악 경험 방지 최소 가이드라인 도출)
        + 5-Fold 교차 검증 (Train 룰 → Test 적용 target_rate/lift 유지 확인)
Step 2: 4점 vs 5점     → RuleFit ("팬심 공략": 보통→충성 고객 한 끗 차이 발굴)

Group C (Q11=5 vs 4 역방향 RuleFit)는 주석 처리.
X_rule: 특수결측 복원·변수 필터만 (MICE 미사용).
"""

'STRICT_ML — 2단계 Q11 만족도 분석 파이프라인.\n\nStep 1: 1~3점 vs 4~5점 → PRIM ("탈출 방지": 최악 경험 방지 최소 가이드라인 도출)\n        + 5-Fold 교차 검증 (Train 룰 → Test 적용 target_rate/lift 유지 확인)\nStep 2: 4점 vs 5점     → RuleFit ("팬심 공략": 보통→충성 고객 한 끗 차이 발굴)\n\nGroup C (Q11=5 vs 4 역방향 RuleFit)는 주석 처리.\nX_rule: 특수결측 복원·변수 필터만 (MICE 미사용).\n'

## 2. 유틸리티 함수 (디렉토리 생성 / CSV 저장)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Utility Functions (defined before any call)
# ═════════════════════════════════════════════════════════════════════════════

def ensure_dirs(*dirs: Path) -> None:
    """디렉토리가 없으면 생성."""
    for d in dirs:
        d.mkdir(parents=True, exist_ok=True)


def save_csv_helper(
    df: pd.DataFrame,
    path: Path,
    log_fn: Callable = None,
    config: Optional[Any] = None,  # Config 타입 (순환 참조 방지)
    force_save: bool = False
) -> None:
    """CSV 저장 + 간단 로그.

    Args:
        df: 저장할 DataFrame
        path: 저장 경로
        log_fn: 로그 함수
        config: Config 객체 (저장 정책 확인용)
        force_save: True이면 저장 정책과 관계없이 강제 저장
    """
    # 저장 정책 확인 (force_save가 True이거나 config가 None이면 항상 저장)
    if not force_save and config is not None:
        if config.save_level == "minimal":
            path_str = str(path)
            # 중간 파일 저장 비활성화 체크
            if "threshold_grid_val_" in path_str and not config.save_threshold_grid:
                return
            if "threshold_candidates_val_" in path_str and not config.save_threshold_candidates:
                return
            if "_threshold_alpha_scenarios" in path_str and not config.save_threshold_alpha_scenarios:
                return
            if "threshold_strategies_summary" in path_str and not config.save_threshold_strategies:
                return
            if "threshold_metrics_full" in path_str and not config.save_threshold_metrics_full:
                return
            if "model_ranking_" in path_str and not config.save_model_rankings:
                return

    ensure_dirs(path.parent)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    msg = f"[Save] {path.name} ({len(df)} rows)"
    if log_fn:
        log_fn(msg)
    else:
        print(msg, flush=True)


# ═════════════════════════════════════════════════════════════════════════════
# Logging
# ═════════════════════════════════════════════════════════════════════════════


## 3. 로깅 (Logger)

In [ ]:

class Logger:
    def __init__(self, verbose: bool = True):
        self.verbose = verbose
        self.logs: List[str] = []

    def log(self, msg: str, level: str = "INFO"):
        timestamp = datetime.now().strftime("%H:%M:%S")
        formatted = f"[{timestamp}] [{level}] {msg}"
        self.logs.append(formatted)
        if self.verbose:
            print(formatted, flush=True)

    def save(self, filepath: Path):
        ensure_dirs(filepath.parent)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write("\n".join(self.logs))


_logger = Logger()


def log(msg: str, level: str = "INFO"):
    _logger.log(msg, level)




## 4. 설정 (Config)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Config
# ═════════════════════════════════════════════════════════════════════════════

# @dataclass
# class Config:
#     data_csv: Path = Path("data.csv")
#     output_dir: Path = Path("./output")
#     target: str = "Q11"
#     demographics: List[str] = field(default_factory=lambda: [
#         "D_MON", "D_NAT", "D_SEX", "D_AGE", "D_MOK", "D_NUM", "D_GUB", "D_BUN"
#     ])
#     random_seed: int = 2026
#     prim_min_support: int = 10
#     # PRIM/RuleFit 전용 X_rule (특수코드 복원·필터만; MICE 없음)
#     prim_rule_missing_rate_threshold: float = 0.5  # 0.3 등으로 실험 시 변경
#     prim_rule_num_missing_code: float = -69.0
#     prim_rule_cat_missing_code: float = -77.0
#     prim_rule_ambiguous_name_substrings: Tuple[str, ...] = (
#         "_dk", "dk", "_DK", "DK",
#     )
#     # 지출/금액류: 음수가 나올 수 있어 비정상 음수 판정에서 제외할 이름 prefix
#     prim_rule_allow_negative_prefixes: Tuple[str, ...] = (
#         "MQ7", "Q7", "KWON", "RQ7", "WQ9", "Q9",
#     )
#     # 다중응답(MQ*)에서 결측 의미가 불명확할 수 있어, 결측률이 이 값 이상이면 제거 (None이면 규칙 비활성)
#     prim_rule_multi_response_missing_drop_threshold: Optional[float] = 0.35
#     # 이진 변수에서 한 범주 비중이 이 값 이상이면 '응답 여부만'에 가깝다고 보고 제거
#     prim_rule_quasi_constant_binary_mass: float = 0.995
#     # 코드북 prefix로 잡히지 않는 변수: 연속형(비율) vs 구간범주(C*) 명시
#     prim_rule_explicit_numerical_features: Tuple[str, ...] = (
#         "Q2_3b2e", "Q2_3b3e", "Q2_3b2d", "Q2_3b3c", "Q2_3b3d", "Q2_3b3b",
#         "Q2_3b2c", "Q2_3b2b", "Q2_3b3a", "Q2_3b2a",
#         "학교숙박60", "민박숙박60", "민박숙박", "학교숙박",
#         "모텔숙박60", "모텔숙박", "게스숙박60", "게스숙박",
#         "친척숙박60", "친척숙박",
#         "한국국외1인대체",
#         "숙박비1인대체", "국제한국1인대체", "국제국외1인대체", "국제수상1인대체",
#         "가이드1인대체", "음식점1인대체", "식음료1인대체",
#         "한국한국1인대체", "한국수상1인대체", "한국철도1인대체", "한국도로1인대체",
#         "대여서1인대체", "유류비1인대체", "문화서1인대체", "오락및1인대체",
#         "쇼핑비1인대체", "데이터1인대체", "치료및1인대체", "미용서1인대체", "기타비1인대체",
#     )
#     prim_rule_explicit_categorical_features: Tuple[str, ...] = (
#         "C한국국외1인대체",
#         "C국제한국1인대체", "C국제국외1인대체", "C국제수상1인대체",
#         "C가이드1인대체", "C음식점1인대체",
#         "C한국한국1인대체", "C한국수상1인대체", "C한국철도1인대체", "C한국도로1인대체",
#         "C대여서1인대체", "C유류비1인대체", "C문화서1인대체", "C오락및1인대체",
#         "C데이터1인대체", "C치료및1인대체", "C미용서1인대체", "C기타비1인대체",
#     )
#     test_size: float = 0.2
#     catboost_threads: int = 4
#     shap_n_sample: int = 2000
#     verbose: bool = True

#     # ============================================================================
#     # 2024 외래관광객조사 코드북 기준 변수 척도 분류 (Scale Classification)
#     # ============================================================================
#     # 명목척도(Nominal) + 서열척도(Ordinal) → 범주형(categorical)
#     # 비율척도(Ratio) → 수치형(numeric)
#     #
#     # prefix 기반 매칭: 변수명이 해당 prefix로 시작하면 해당 척도로 분류
#     # exact name 매칭: prefix로 분류 불가한 변수는 정확한 이름으로 매칭
#     # ============================================================================
#     nominal_prefixes: Tuple[str, ...] = (
#         # 기본 식별/인구통계: D_SEX, D_NAT, D_NAT2, D_MON
#         "D_SEX", "D_NAT", "D_MON",
#         # 여행 형태/방한 목적: TYP, D_GUB, Q1, D_MOK, Q6
#         "TYP", "D_GUB", "D_MOK",
#         # 동반자 유형: Q7A, Q7a_dk~Q7a8
#         "Q7A", "Q7a",
#         # 관광 행태/선택 요인 (다중/우선순위): Q1_1a, Q3_1a, Q3_2a
#         "Q1_1a", "Q3_1a", "Q3_2a",
#         # 참여활동/만족활동: Q8a, Q8_1a
#         "Q8a", "Q8_1a",
#         # 정보수집/예약: Q4a, Q4_1a, Q4_2a, Q5_1a
#         "Q4a", "Q4_1a", "Q4_2a", "Q5_1a",
#         # 이동동선 (방문국가/지역): Q2a, Q2_1a, Q2_3a, ZQ2
#         "Q2a", "Q2_1a", "Q2_3a", "ZQ2",
#         # 방문지역/가장좋았던곳: Q9_1_, Q9_2a
#         "Q9_1_", "Q9_2a",
#         # 방문권역: KWON
#         "KWON",
#         # 숙박시설 시도별/유형: WQ9_5a, Q9_5A
#         "WQ9_5a", "Q9_5A",
#         # 쇼핑: Q10_2a, Q10_3a
#         "Q10_2a", "Q10_3a",
#     )
#     nominal_exact: Tuple[str, ...] = (
#         "pnid", "Q1", "Q6", "Q7A",
#     )
#     ordinal_prefixes: Tuple[str, ...] = (
#         # 만족도/행동의도 (리커트): Q12a
#         "Q12a",
#         # 방문횟수 범주화: RVIT, XRVIT, D_NUM
#         "RVIT", "XRVIT",
#         # 체류기간 범주화: R1일HAP
#         "R1일HAP",
#         # 동반자수 범주화: RQ7
#         "RQ7",
#         # 지출경비 범주화 (금액 구간): C총액, C여행사, C단기투어, RDAY, C교통, C숙박, C식음료, C쇼핑
#         "C총액", "C여행사", "C단기투어", "RDAY", "C교통", "C숙박", "C식음료", "C쇼핑",
#     )
#     ordinal_exact: Tuple[str, ...] = (
#         "Q11", "Q13", "Q14", "Q5", "D_AGE", "D_BUN", "D_NUM",
#         "Q6_1R",
#     )
#     ratio_prefixes: Tuple[str, ...] = (
#         # 방문/체류 실측값 (일/박, 횟수): MVIT, M박HAP, M일HAP
#         "MVIT", "M박HAP", "M일HAP",
#         # 시도별 숙박/체재: 서울박, 서울일, 부산박, 부산일, ...
#         "서울박", "서울일", "부산박", "부산일", "대구박", "대구일",
#         "인천박", "인천일", "광주박", "광주일", "대전박", "대전일",
#         "울산박", "울산일", "세종박", "세종일",
#         "경기박", "경기일", "강원박", "강원일",
#         "충북박", "충북일", "충남박", "충남일",
#         "전북박", "전북일", "전남박", "전남일",
#         "경북박", "경북일", "경남박", "경남일",
#         "제주박", "제주일",
#         # 숙박시설별 숙박일수: 호텔숙박, 콘도숙박, 게스트하우스숙박, ...
#         "호텔숙박", "콘도숙박", "게스트하우스숙박", "기타숙박",
#         # 동반자수 실측: MQ7
#         "MQ7",
#         # 지출경비 절대금액: 총액1인, MDAY
#         "총액1인", "MDAY",
#         # 세부항목 지출금액 (한글): 여행사1인, 교통1인, 숙박1인, 식음료1인, 쇼핑1인
#         "여행사1인", "단기투어상품1인",
#         # 쇼핑비중: MQ10_2b
#         "MQ10_2b",
#         # 가중치: weight
#         "weight",
#     )
#     ratio_exact: Tuple[str, ...] = (
#         "Q6_1",
#     )
#     # ============================================================================
#     # 저장 정책 플래그 (SAVE_LEVEL)
#     # ============================================================================
#     # "minimal": 최소한의 최종 요약 파일만 저장
#     #   - model_metrics_validation.csv, model_metrics_full.csv
#     #   - metric_best_models.json, metric_best_models.csv
#     #   - validation 기준 지표별 최고 모델에 대한 PRIM/SHAP/CPI 결과만 저장
#     # "full": 모든 중간 결과 포함 (디버깅용)
#     save_level: str = "minimal"  # "minimal" 또는 "full"

#     # 저장 허용 파일 패턴 (save_level="minimal"일 때만 적용)
#     # True: 저장 허용, False: 저장 비활성화
#     save_threshold_grid: bool = False  # threshold_grid_val_*.csv
#     save_threshold_candidates: bool = False  # threshold_candidates_val_*.csv
#     save_threshold_alpha_scenarios: bool = False  # *_threshold_alpha_scenarios.csv
#     save_threshold_strategies: bool = False  # threshold_strategies_summary.csv
#     save_threshold_metrics_full: bool = False  # threshold_metrics_full.csv
#     save_model_rankings: bool = False  # model_ranking_*.csv
#     save_all_models_prim: bool = False  # 모든 모델의 PRIM 결과
#     save_all_models_prim_subgroups: bool = False  # 모든 모델의 PRIM subgroups 통합

#     def __post_init__(self):
#         self.data_csv = Path(self.data_csv)
#         self.output_dir = Path(self.output_dir)
#         self.output_dir.mkdir(parents=True, exist_ok=True)


## 5. 데이터 정제 유틸리티

코드북 기준 척도 분류 → 특수결측코드 복원 → 필터링 → float 행렬 구성

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Data Cleaning Utilities
# ═════════════════════════════════════════════════════════════════════════════

_SAFE_CLIP_ABS: float = 1e15
_FLOAT_DTYPE: str = "float64"

# 최소지지도 스윕 값(고정). 실제 CSV 저장은 `save_prim_min_support_sweep_csvs()`에서 수행.
PRIM_MIN_SUPPORT_SWEEP: Tuple[int, ...] = tuple(range(10, 101, 10))


def sanitize_numeric_matrix(
    X: pd.DataFrame, *, clip_abs: float = _SAFE_CLIP_ABS,
    fill_nan: Optional[float] = None,
) -> pd.DataFrame:
    X_out = X.copy()
    num_cols = X_out.select_dtypes(include=[np.number]).columns
    if len(num_cols) == 0:
        return X_out
    X_out[num_cols] = X_out[num_cols].astype(_FLOAT_DTYPE)
    X_out[num_cols] = X_out[num_cols].replace([np.inf, -np.inf], np.nan)
    X_out[num_cols] = X_out[num_cols].clip(-clip_abs, clip_abs)
    if fill_nan is not None:
        X_out[num_cols] = X_out[num_cols].fillna(fill_nan)
    return X_out


def _classify_variable_scale(col: str, config: Config) -> Tuple[str, str]:
    """코드북 기준으로 변수의 척도(scale_type)와 데이터 유형(data_type)을 분류.

    Returns:
        (scale_type, data_type)
        scale_type: "nominal" | "ordinal" | "ratio" | "unknown"
        data_type:  "categorical" | "numeric" | "unknown"
    """
    if col in config.prim_rule_explicit_categorical_features:
        # 구간 카테고리(98/99 등): 서열(순서 있는 구간) — PRIM peeling 대상은 아래 _prim_cat_cols에서 지정
        return "ordinal", "categorical"
    if col in config.prim_rule_explicit_numerical_features:
        return "ratio", "numeric"
    if col in config.nominal_exact or any(col.startswith(p) for p in config.nominal_prefixes):
        return "nominal", "categorical"
    if col in config.ordinal_exact or any(col.startswith(p) for p in config.ordinal_prefixes):
        return "ordinal", "categorical"
    if col in config.ratio_exact or any(col.startswith(p) for p in config.ratio_prefixes):
        return "ratio", "numeric"
    return "unknown", "unknown"


def _prim_rule_restore_column(
    s: pd.Series,
    *,
    num_code: float,
    cat_code: float,
    scale_type: str = "unknown",
) -> Tuple[pd.Series, str]:
    """척도 분류를 반영하여 특수결측코드를 NaN으로 복원.

    - ratio(수치형): num_code(-69)만 NaN 복원
    - nominal/ordinal(범주형): cat_code(-77) + num_code(-69) 모두 NaN 복원
    - unknown: dtype 기반 fallback (기존 로직)
    """
    v = pd.to_numeric(s, errors="coerce").astype(float)

    if scale_type == "ratio":
        m = np.isclose(v, float(num_code), rtol=0.0, atol=1e-6, equal_nan=False)
        v = v.where(~m, np.nan)
        return v, "numeric"

    if scale_type in ("nominal", "ordinal"):
        m77 = np.isclose(v, float(cat_code), rtol=0.0, atol=1e-6, equal_nan=False)
        m69 = np.isclose(v, float(num_code), rtol=0.0, atol=1e-6, equal_nan=False)
        v = v.where(~(m77 | m69), np.nan)
        return v, "categorical"

    # unknown: dtype 기반 fallback
    if pd.api.types.is_numeric_dtype(s):
        m = np.isclose(v, float(num_code), rtol=0.0, atol=1e-6, equal_nan=False)
        v = v.where(~m, np.nan)
        return v, "numeric"
    m77 = np.isclose(v, float(cat_code), rtol=0.0, atol=1e-6, equal_nan=False)
    m69 = np.isclose(v, float(num_code), rtol=0.0, atol=1e-6, equal_nan=False)
    v = v.where(~(m77 | m69), np.nan)
    return v, "categorical"


def _col_allows_negative(name: str, prefixes: Tuple[str, ...]) -> bool:
    return any(name.startswith(p) for p in prefixes)


def build_x_rule_for_prim_rulefit(
    df: pd.DataFrame,
    exclude_cols: Set[str],
    config: Config,
    *,
    missing_rate_threshold: Optional[float] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """PRIM/RuleFit용 행렬: 코드북 척도분류 → 특수결측코드 복원 → 필터 → float 행렬."""
    thr = float(
        missing_rate_threshold
        if missing_rate_threshold is not None
        else config.prim_rule_missing_rate_threshold
    )
    num_code = float(config.prim_rule_num_missing_code)
    cat_code = float(config.prim_rule_cat_missing_code)
    amb_subs = tuple(config.prim_rule_ambiguous_name_substrings)
    neg_ok = tuple(config.prim_rule_allow_negative_prefixes)
    mq_thr = getattr(config, "prim_rule_multi_response_missing_drop_threshold", None)

    candidate_cols = [c for c in df.columns if c not in exclude_cols]
    restored: Dict[str, pd.Series] = {}
    role_map: Dict[str, str] = {}
    scale_map: Dict[str, str] = {}
    dtype_map: Dict[str, str] = {}

    for col in candidate_cols:
        scale_type, data_type = _classify_variable_scale(col, config)
        scale_map[col] = scale_type
        dtype_map[col] = data_type
        s_raw = df[col]
        v, role = _prim_rule_restore_column(
            s_raw, num_code=num_code, cat_code=cat_code,
            scale_type=scale_type,
        )
        v.index = df.index
        restored[col] = v
        role_map[col] = role

    report_rows: List[Dict[str, Any]] = []
    remove_reasons: Dict[str, List[str]] = {c: [] for c in candidate_cols}

    for col in candidate_cols:
        v = restored[col]
        reasons: List[str] = []
        miss_rate = float(v.isna().mean()) if len(v) else 1.0

        cl = col.lower()
        if any(sub.lower() in cl for sub in amb_subs):
            reasons.append("ambiguous_name")

        if miss_rate >= thr:
            reasons.append(f"high_missing(>={thr:.2f})")

        vn = v.dropna()
        if len(vn) == 0:
            reasons.append("all_missing_after_restore")
        elif int(vn.nunique()) <= 1:
            reasons.append("constant_after_restore")

        if len(vn) > 0 and int(vn.nunique()) == 2:
            prop = float(vn.value_counts(normalize=True).iloc[0])
            qmass = float(getattr(config, "prim_rule_quasi_constant_binary_mass", 0.995))
            if prop >= qmass or prop <= (1.0 - qmass):
                reasons.append("quasi_constant_binary")

        if mq_thr is not None and col.startswith("MQ") and miss_rate >= float(mq_thr):
            reasons.append("multi_response_high_missing")

        if len(vn) > 0 and not _col_allows_negative(col, neg_ok):
            if bool((vn < -1e-6).any()):
                reasons.append("abnormal_negative_values")

        remove_reasons[col] = reasons
        report_rows.append(
            {
                "variable": col,
                "scale_type": scale_map[col],
                "data_type": dtype_map[col],
                "role_original": role_map[col],
                "n_unique": int(vn.nunique()) if len(vn) else 0,
                "missing_rate_after_restore": round(miss_rate, 6),
                "removed": bool(reasons),
                "removal_reason": "; ".join(reasons) if reasons else "",
                "in_final_x_rule": not bool(reasons),
            }
        )

    final_cols = [c for c in candidate_cols if not remove_reasons[c]]
    X_rule = pd.concat([restored[c] for c in final_cols], axis=1)
    X_rule.columns = final_cols
    X_rule = sanitize_numeric_matrix(X_rule, fill_nan=None)
    X_rule = X_rule.astype(float)

    for c in X_rule.columns:
        s = X_rule[c].dropna()
        if s.empty:
            continue
        if np.any(np.isclose(s.values, num_code, rtol=0.0, atol=1e-6)):
            raise ValueError(f"X_rule column {c!r} still contains numeric missing code {num_code}")
        if np.any(np.isclose(s.values, cat_code, rtol=0.0, atol=1e-6)):
            raise ValueError(f"X_rule column {c!r} still contains categorical missing code {cat_code}")

    report_df = pd.DataFrame(report_rows).sort_values(
        ["removed", "missing_rate_after_restore"], ascending=[False, False]
    ).reset_index(drop=True)

    # 최종 변수의 범주형/수치형 분류 집합
    final_categorical_cols = [
        c for c in final_cols if dtype_map.get(c, "unknown") == "categorical"
    ]
    final_numeric_cols = [
        c for c in final_cols if dtype_map.get(c, "unknown") == "numeric"
    ]
    final_unknown_cols = [
        c for c in final_cols if dtype_map.get(c, "unknown") == "unknown"
    ]

    removed_vars = [c for c in candidate_cols if remove_reasons[c]]
    meta: Dict[str, Any] = {
        "missing_rate_threshold": thr,
        "n_candidate": len(candidate_cols),
        "n_removed": len(removed_vars),
        "n_final": len(final_cols),
        "removed_variables": removed_vars,
        "final_variables": final_cols,
        "num_missing_code": num_code,
        "cat_missing_code": cat_code,
        "categorical_cols": final_categorical_cols,
        "numeric_cols": final_numeric_cols,
        "unknown_cols": final_unknown_cols,
        "scale_map": {c: scale_map[c] for c in final_cols},
        "dtype_map": {c: dtype_map[c] for c in final_cols},
    }
    return X_rule, report_df, meta




## 6. PRIM (Patient Rule Induction Method)

- 수치형: quantile 기반 peeling
- 범주형: 카테고리별 peeling (target_rate 가장 낮은 범주 제거)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# PRIM (Patient Rule Induction Method) — Segment Exploration Only
# ═════════════════════════════════════════════════════════════════════════════

@dataclass
class PrimConfig:
    """PRIM 알고리즘 설정."""
    peeling_fraction: float = 0.05  # alpha: 각 peeling 단계에서 제거할 데이터 비율
    min_support: int = 50  # 최소 박스 크기 (절대값)
    min_support_ratio: float = 0.01  # 최소 박스 크기 (전체 대비 비율)
    max_boxes: int = 10  # 최대 생성할 박스 수
    min_improvement: float = 0.0  # 최소 개선량 (gain threshold)
    max_peeling_iterations: int = 50  # 각 박스당 최대 peeling 반복 횟수


def run_prim_for_target(
    X: pd.DataFrame,
    y_target: np.ndarray,
    config: PrimConfig,
    feature_subset: Optional[List[str]] = None,
    categorical_cols: Optional[Set[str]] = None,
) -> List[Dict[str, Any]]:
    """PRIM 알고리즘을 특정 타겟에 대해 실행.

    수치형(ratio) 변수: quantile 기반 peeling (상/하위 alpha 분위 제거)
    범주형(nominal/ordinal) 변수: 카테고리별 제거 peeling (target_rate 가장 낮은 범주 제거)

    Args:
        X: Feature DataFrame (Q11 제외한 설명변수들)
        y_target: 이진 타겟 (예: y_very_low, y_almost_top)
        config: PRIM 설정
        feature_subset: 사용할 변수 목록 (None이면 모든 수치형 변수 사용)
        categorical_cols: 범주형으로 처리할 변수 이름 집합 (코드북 기반)
    """
    if categorical_cols is None:
        categorical_cols = set()

    if feature_subset is not None:
        feature_subset = [f for f in feature_subset if f in X.columns]
        X_work = X[feature_subset].copy()
    else:
        X_work = X.select_dtypes(include=[np.number]).copy()

    y_arr = np.asarray(y_target, dtype=float)
    if len(y_arr) == 0:
        return []

    pop_rate = float(y_arr.mean()) if len(y_arr) > 0 else 0.0
    if pop_rate == 0.0:
        return []

    min_support_abs = max(
        config.min_support,
        int(config.min_support_ratio * len(y_arr))
    )

    _CAT_MAX_UNIQUE = 30  # 고유값이 이보다 많으면 수치형 peeling으로 fallback

    segments: List[Dict[str, Any]] = []
    mask = np.ones(len(y_arr), dtype=bool)

    for box_id in range(config.max_boxes):
        X_sub = X_work.loc[mask]
        y_sub = y_arr[mask]

        if len(y_sub) < min_support_abs * 2:
            break

        current_mean = float(y_sub.mean())
        rules: List[str] = []
        local_mask = np.ones(len(y_sub), dtype=bool)

        for _ in range(config.max_peeling_iterations):
            best_gain = -np.inf
            best_rule = None
            best_new_mask = None

            n = int(local_mask.sum())
            if n < min_support_abs:
                break

            for col in X_sub.columns:
                vals = X_sub[col].values[local_mask]
                y_local = y_sub[local_mask]

                if np.all(np.isnan(vals)):
                    continue

                valid = ~np.isnan(vals)
                if valid.sum() < min_support_abs:
                    continue

                is_cat = (
                    col in categorical_cols
                    and int(np.unique(vals[valid]).size) <= _CAT_MAX_UNIQUE
                )

                if is_cat:
                    # -- 범주형 peeling: target_rate 가장 낮은 범주 하나 제거 --
                    uniq = np.unique(vals[valid])
                    for cat_val in uniq:
                        candidate_mask = local_mask.copy()
                        candidate_mask[local_mask] &= ~(
                            np.isclose(vals, cat_val, rtol=0, atol=1e-9) & valid
                        )
                        if candidate_mask.sum() < min_support_abs:
                            continue
                        new_mean = float(y_sub[candidate_mask].mean())
                        gain = new_mean - current_mean
                        if gain > best_gain and gain >= config.min_improvement:
                            kept_vals = np.unique(vals[valid & ~np.isclose(vals, cat_val, rtol=0, atol=1e-9)])
                            if kept_vals.size == 1:
                                best_rule_text = f"{col} == {kept_vals[0]:.4g}"
                            else:
                                # 전체 고유값 대비 제외된 값(여집합)이 적으면 not in 으로 표현
                                all_vals = np.unique(X_sub[col].dropna().values)
                                excluded = np.setdiff1d(all_vals, kept_vals)
                                if excluded.size > 0 and excluded.size <= 6 and kept_vals.size > 6:
                                    exc_txt = ", ".join(f"{v:.4g}" for v in excluded)
                                    best_rule_text = f"{col} not in {{{exc_txt}}}"
                                else:
                                    kept_txt = ", ".join(f"{v:.4g}" for v in kept_vals[:6])
                                    if kept_vals.size > 6:
                                        kept_txt += ", ..."
                                    best_rule_text = f"{col} in {{{kept_txt}}}"
                            best_gain = gain
                            best_rule = best_rule_text
                            best_new_mask = candidate_mask
                else:
                    # -- 수치형 peeling: 상/하위 alpha-fraction 제거 --
                    peel_n = max(1, int(config.peeling_fraction * valid.sum()))
                    sorted_idx = np.argsort(vals[valid])
                    for direction in ("low", "high"):
                        candidate_mask = local_mask.copy()
                        if direction == "low":
                            threshold = vals[valid][sorted_idx[peel_n]]
                            candidate_mask[local_mask] &= ~(
                                (vals <= threshold) & valid
                            )
                            rule_str = f"{col} > {threshold:.4g}"
                        else:
                            threshold = vals[valid][sorted_idx[-(peel_n + 1)]]
                            candidate_mask[local_mask] &= ~(
                                (vals >= threshold) & valid
                            )
                            rule_str = f"{col} < {threshold:.4g}"

                        if candidate_mask.sum() < min_support_abs:
                            continue

                        new_mean = float(y_sub[candidate_mask].mean())
                        gain = new_mean - current_mean

                        if gain > best_gain and gain >= config.min_improvement:
                            best_gain = gain
                            best_rule = rule_str
                            best_new_mask = candidate_mask

            if best_rule is None or best_gain <= config.min_improvement:
                break

            local_mask = best_new_mask
            current_mean = float(y_sub[local_mask].mean())
            rules.append(best_rule)

        n_segment = int(local_mask.sum())
        if n_segment < min_support_abs or not rules:
            break

        target_rate = float(y_sub[local_mask].mean())

        n_target_total = int(y_arr.sum())
        n_target_in_subgroup = int(y_sub[local_mask].sum())
        coverage = n_target_in_subgroup / n_target_total if n_target_total > 0 else 0.0

        lift = target_rate / pop_rate if pop_rate > 0 else np.nan

        segments.append({
            "rules": rules,
            "n_subgroup": n_segment,
            "target_rate": target_rate,
            "coverage": coverage,
            "lift": lift,
        })

        indices_in_sub = np.where(mask)[0]
        remove_idx = indices_in_sub[local_mask]
        mask[remove_idx] = False

    return segments


def summarize_prim_results(
    prim_results_list: List[Dict[str, Any]],
    target_label: str,
) -> pd.DataFrame:
    """PRIM 결과를 요약하여 DataFrame으로 변환.

    Args:
        prim_results_list: run_prim_for_target의 반환값
        target_label: 타겟 레이블 (예: 'very_low', 'almost_top')

    Returns:
        DataFrame with columns:
        - 'target_label': 타겟 레이블
        - 'subgroup_id': 서브그룹 ID (1부터 시작)
        - 'rule_text': 규칙 텍스트 (AND로 연결)
        - 'n_subgroup': 서브그룹 샘플 수
        - 'target_rate': 서브그룹 내 타겟 비율
        - 'coverage': 전체 타겟 중 이 서브그룹이 차지하는 비율
        - 'lift': target_rate / 전체 target_rate
    """
    rows = []
    for idx, seg in enumerate(prim_results_list):
        rows.append({
            "target_label": target_label,
            "subgroup_id": idx + 1,
            "rule_text": " AND ".join(seg["rules"]),
            "n_subgroup": seg["n_subgroup"],
            "target_rate": round(seg["target_rate"], 4),
            "coverage": round(seg["coverage"], 4),
            "lift": round(seg["lift"], 4) if not np.isnan(seg["lift"]) else np.nan,
        })

    return pd.DataFrame(rows)


_PRIM_SWEEP_RESULT_COLS: List[str] = [
    "target_label", "min_support", "subgroup_id", "rule_id", "rule_text",
    "n_subgroup", "target_rate", "coverage", "lift", "method",
]


def save_prim_min_support_sweep_csvs(
    X_g: pd.DataFrame,
    y_g: np.ndarray,
    *,
    group_key: str,
    target_label: str,
    tables_dir: Path,
    config: Config,
    categorical_cols: Optional[Set[str]] = None,
) -> None:
    """`PRIM_MIN_SUPPORT_SWEEP` 각 값에 대해 PRIM을 돌리고 `tables_dir`에 CSV로 저장한다.

    파일명: ``{group_key}_prim_results_ms{min_support}.csv`` (예: ``group_a_prim_results_ms30.csv``).
    """
    ensure_dirs(tables_dir)
    for ms in PRIM_MIN_SUPPORT_SWEEP:
        prim_cfg_ms = PrimConfig(
            peeling_fraction=0.05,
            min_support=int(ms),
            min_support_ratio=0.0,
            max_boxes=10,
            min_improvement=0.0,
            max_peeling_iterations=50,
        )
        prim_raw_ms = run_prim_for_target(
            X_g, y_g, prim_cfg_ms, feature_subset=None,
            categorical_cols=categorical_cols,
        )
        prim_df_ms = summarize_prim_results(prim_raw_ms, target_label)
        if not prim_df_ms.empty:
            prim_df_ms = prim_df_ms.copy()
            prim_df_ms["method"] = "PRIM"
            prim_df_ms["rule_id"] = prim_df_ms["subgroup_id"]
            prim_df_ms.insert(1, "min_support", int(ms))
            prim_df_ms = prim_df_ms[_PRIM_SWEEP_RESULT_COLS]
        else:
            prim_df_ms = pd.DataFrame(columns=_PRIM_SWEEP_RESULT_COLS)

        out_path = tables_dir / f"{group_key}_prim_results_ms{int(ms)}.csv"
        save_csv_helper(
            prim_df_ms,
            out_path,
            log,
            config=config,
            force_save=True,
        )
        log(
            f"[{group_key.upper()}][PRIM sweep] wrote {out_path.name} "
            f"(min_support={ms}, n_rows={len(prim_df_ms)})"
        )




## 7. PRIM 5-Fold 교차 검증

Train 세트에서 PRIM 룰 추출 → Test 세트 적용으로 target_rate·lift 유지 검증

In [ ]:
def cross_validate_prim(
    X: pd.DataFrame,
    y: np.ndarray,
    prim_cfg: PrimConfig,
    *,
    n_folds: int = 5,
    random_state: int = 2026,
    target_label: str = "",
    categorical_cols: Optional[Set[str]] = None,
    log_fn: Callable = None,
) -> pd.DataFrame:
    """PRIM 5-Fold 교차 검증.

    각 Fold의 Train 세트에서 PRIM 룰을 추출하고,
    해당 룰을 Test 세트에 적용하여 target_rate·lift가 유지되는지 검증.

    Returns:
        DataFrame: fold, rule_id, rule_text, train_target_rate, test_target_rate,
                   train_lift, test_lift, train_n, test_n
    """
    _log = log_fn or log
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    cv_rows: List[Dict[str, Any]] = []

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        _log(f"[CV Fold {fold_idx}/{n_folds}] train={len(y_train)}, test={len(y_test)}, "
             f"train_pos_rate={y_train.mean():.4f}, test_pos_rate={y_test.mean():.4f}")

        # Train에서 PRIM 룰 추출
        prim_raw = run_prim_for_target(
            X_train, y_train, prim_cfg,
            feature_subset=None,
            categorical_cols=categorical_cols,
        )
        prim_df = summarize_prim_results(prim_raw, target_label)
        if prim_df.empty:
            _log(f"[CV Fold {fold_idx}] No rules found in train set", level="WARN")
            continue

        prim_df["rule_text"] = "PRIM_" + prim_df["rule_text"].astype(str)

        # 각 룰을 Test에 적용
        for _, row in prim_df.iterrows():
            rule_text = row["rule_text"]
            train_stats = apply_prim_rule_to_data(X_train, y_train, rule_text)
            test_stats = apply_prim_rule_to_data(X_test, y_test, rule_text)

            cv_rows.append({
                "fold": fold_idx,
                "min_support": prim_cfg.min_support, # Add min_support here
                "rule_id": int(row["subgroup_id"]),
                "rule_text": rule_text,
                "train_n": train_stats["n_subgroup"],
                "train_target_rate": train_stats["target_rate"],
                "train_lift": train_stats["lift"],
                "train_coverage": train_stats["coverage"],
                "test_n": test_stats["n_subgroup"],
                "test_target_rate": test_stats["target_rate"],
                "test_lift": test_stats["lift"],
                "test_coverage": test_stats["coverage"],
            })

    if not cv_rows:
        return pd.DataFrame()

    cv_df = pd.DataFrame(cv_rows)

    # Fold별 target_rate 평균·표준편차 출력
    _log("\n" + "=" * 70)
    _log(f"[CV Summary] {target_label} — {n_folds}-Fold Cross-Validation Results")
    _log("=" * 70)

    fold_summary = cv_df.groupby("fold").agg(
        n_rules=("rule_id", "count"),
        train_rate_mean=("train_target_rate", "mean"),
        train_rate_std=("train_target_rate", "std"),
        test_rate_mean=("test_target_rate", "mean"),
        test_rate_std=("test_target_rate", "std"),
        train_lift_mean=("train_lift", "mean"),
        test_lift_mean=("test_lift", "mean"),
    ).reset_index()

    for _, fs in fold_summary.iterrows():
        _log(
            f"  Fold {int(fs['fold'])}: rules={int(fs['n_rules'])}, "
            f"train_rate={fs['train_rate_mean']:.4f}±{fs['train_rate_std']:.4f}, "
            f"test_rate={fs['test_rate_mean']:.4f}±{fs['test_rate_std']:.4f}, "
            f"train_lift={fs['train_lift_mean']:.4f}, test_lift={fs['test_lift_mean']:.4f}"
        )

    # 전체 요약
    overall_train_mean = cv_df["train_target_rate"].mean()
    overall_train_std = cv_df["train_target_rate"].std()
    overall_test_mean = cv_df["test_target_rate"].mean()
    overall_test_std = cv_df["test_target_rate"].std()
    _log(f"\n  [Overall] train_target_rate: {overall_train_mean:.4f} ± {overall_train_std:.4f}")
    _log(f"  [Overall] test_target_rate:  {overall_test_mean:.4f} ± {overall_test_std:.4f}")
    _log(f"  [Overall] rate_drop (train→test): {overall_train_mean - overall_test_mean:.4f}")
    _log("=" * 70)

    return cv_df

## 8. RuleFit 5-Fold 교차 검증

Train 세트에서 Multi-Booster ElasticNet RuleFit 룰 추출 → Test 세트 적용으로 target_rate·lift 유지 검증

- 각 Fold에서 GBM / XGB / LGBM / CatBoost 4개 부스터 병렬 실행
- ElasticNet으로 선택된 규칙을 Test 세트에 직접 적용

In [ ]:
def cross_validate_rulefit(
    X: pd.DataFrame,
    y: np.ndarray,
    *,
    n_folds: int = 5,
    random_state: int = 2026,
    target_label: str = "",
    min_support: int = 50,
    max_rules: int = 30,
    n_estimators: int = 200,
    max_depth: int = 3,
    log_fn: Callable = None,
) -> pd.DataFrame:
    """RuleFit(Multi-Booster ElasticNet) 5-Fold 교차 검증.

    각 Fold의 Train 세트에서 RuleFit 룰을 추출하고,
    해당 룰을 Test 세트에 적용하여 target_rate·lift가 유지되는지 검증.

    Returns:
        DataFrame: fold, rule_id, rule_text, booster, importance,
                   train_n, train_target_rate, train_lift, train_coverage,
                   test_n, test_target_rate, test_lift, test_coverage
    """
    _log = log_fn or log
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    cv_rows: List[Dict[str, Any]] = []

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        _log(f"[RuleFit CV Fold {fold_idx}/{n_folds}] train={len(y_train)}, test={len(y_test)}, "
             f"train_pos_rate={y_train.mean():.4f}, test_pos_rate={y_test.mean():.4f}")

        _ms = max(min_support, int(len(y_train) * 0.01))
        _msl = max(min_support // 2, int(len(y_train) * 0.005))

        # Train에서 Multi-Booster RuleFit 룰 추출
        multi_results = run_multi_booster_rulefit(
            X_train, y_train,
            boosters=["GBM", "XGB", "LGBM", "CB"],
            min_support=_ms,
            max_rules=max_rules,
            random_state=random_state,
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=_msl,
            subsample=0.8,
            l1_ratio=0.6,
            cv_folds=5,
            include_linear=True,
            n_jobs=-1,
        )

        rulefit_output = merge_multi_booster_results(multi_results, max_rules=max_rules)
        rulefit_df = summarize_rulefit_en_results(rulefit_output, target_label)

        if rulefit_df.empty:
            _log(f"[RuleFit CV Fold {fold_idx}] No rules found in train set", level="WARN")
            continue

        # 각 룰을 Train/Test에 적용
        for _, row in rulefit_df.iterrows():
            rule_text = str(row["rule_text"])
            train_stats = apply_prim_rule_to_data(X_train, y_train, rule_text)
            test_stats = apply_prim_rule_to_data(X_test, y_test, rule_text)

            cv_rows.append({
                "fold": fold_idx,
                "rule_id": int(row["rule_id"]),
                "rule_text": rule_text,
                "booster": row.get("booster", ""),
                "importance": round(float(row.get("importance", 0.0)), 6),
                "train_n": train_stats["n_subgroup"],
                "train_target_rate": train_stats["target_rate"],
                "train_lift": train_stats["lift"],
                "train_coverage": train_stats["coverage"],
                "test_n": test_stats["n_subgroup"],
                "test_target_rate": test_stats["target_rate"],
                "test_lift": test_stats["lift"],
                "test_coverage": test_stats["coverage"],
            })

    if not cv_rows:
        return pd.DataFrame()

    cv_df = pd.DataFrame(cv_rows)

    # Fold별 target_rate 평균·표준편차 출력
    _log("\n" + "=" * 70)
    _log(f"[RuleFit CV Summary] {target_label} — {n_folds}-Fold Cross-Validation Results")
    _log("=" * 70)

    fold_summary = cv_df.groupby("fold").agg(
        n_rules=("rule_id", "count"),
        train_rate_mean=("train_target_rate", "mean"),
        train_rate_std=("train_target_rate", "std"),
        test_rate_mean=("test_target_rate", "mean"),
        test_rate_std=("test_target_rate", "std"),
        train_lift_mean=("train_lift", "mean"),
        test_lift_mean=("test_lift", "mean"),
    ).reset_index()

    for _, fs in fold_summary.iterrows():
        _log(
            f"  Fold {int(fs['fold'])}: rules={int(fs['n_rules'])}, "
            f"train_rate={fs['train_rate_mean']:.4f}±{fs['train_rate_std']:.4f}, "
            f"test_rate={fs['test_rate_mean']:.4f}±{fs['test_rate_std']:.4f}, "
            f"train_lift={fs['train_lift_mean']:.4f}, test_lift={fs['test_lift_mean']:.4f}"
        )

    overall_train_mean = cv_df["train_target_rate"].mean()
    overall_train_std = cv_df["train_target_rate"].std()
    overall_test_mean = cv_df["test_target_rate"].mean()
    overall_test_std = cv_df["test_target_rate"].std()
    _log(f"\n  [Overall] train_target_rate: {overall_train_mean:.4f} ± {overall_train_std:.4f}")
    _log(f"  [Overall] test_target_rate:  {overall_test_mean:.4f} ± {overall_test_std:.4f}")
    _log(f"  [Overall] rate_drop (train→test): {overall_train_mean - overall_test_mean:.4f}")
    _log("=" * 70)

    return cv_df




## 9. RuleFit 헬퍼 및 변수 축 요약

RandomForest 트리 경로 기반 규칙 추출 + PRIM/RuleFit 핵심 변수 빈도 요약

In [ ]:
def run_rulefit_for_target(
    X: pd.DataFrame,
    y_target: np.ndarray,
    min_support: int = 10,
    max_rules: int = 20,
    random_state: int = 42,
) -> List[Dict[str, Any]]:
    """RuleFit-style rule extraction using tree-path rules.

    규칙(AND 조건)을 트리 경로에서 추출하고, 각 규칙의 lift/coverage 및
    간단한 중요도 점수를 계산합니다.
    """
    X_num = X.select_dtypes(include=[np.number]).copy()
    if X_num.empty:
        return []
    X_num = X_num.fillna(X_num.median(numeric_only=True))
    y_arr = np.asarray(y_target, dtype=int)
    if len(y_arr) == 0 or y_arr.sum() == 0:
        return []

    pop_rate = float(y_arr.mean())
    n_target_total = int(y_arr.sum())

    rf = RandomForestClassifier(
        n_estimators=120,
        max_depth=3,
        min_samples_leaf=max(2, min_support // 2),
        random_state=random_state,
        n_jobs=1,
        class_weight="balanced_subsample",
    )
    rf.fit(X_num, y_arr)

    feature_names = X_num.columns.tolist()
    rule_candidates: List[List[Tuple[str, str, float]]] = []

    for est in rf.estimators_:
        tree = est.tree_
        children_left = tree.children_left
        children_right = tree.children_right
        features = tree.feature
        thresholds = tree.threshold

        def _traverse(node: int, path: List[Tuple[str, str, float]]) -> None:
            if children_left[node] == children_right[node]:
                if path:
                    rule_candidates.append(path.copy())
                return
            feat_idx = int(features[node])
            if feat_idx < 0 or feat_idx >= len(feature_names):
                return
            feat = feature_names[feat_idx]
            thr = float(thresholds[node])
            _traverse(children_left[node], path + [(feat, "<=", thr)])
            _traverse(children_right[node], path + [(feat, ">", thr)])

        _traverse(0, [])

    # 중복 규칙 제거
    unique_rules: Dict[str, List[Tuple[str, str, float]]] = {}
    for conds in rule_candidates:
        text = " AND ".join([f"{f} {op} {thr:.4g}" for f, op, thr in conds])
        unique_rules.setdefault(text, conds)

    rows: List[Dict[str, Any]] = []
    for rule_text, conds in unique_rules.items():
        mask = np.ones(len(X_num), dtype=bool)
        for feat, op, thr in conds:
            vals = X_num[feat].values
            if op == "<=":
                mask &= vals <= thr
            else:
                mask &= vals > thr
        n_subgroup = int(mask.sum())
        if n_subgroup < min_support:
            continue
        y_sub = y_arr[mask]
        if len(y_sub) == 0:
            continue
        target_rate = float(y_sub.mean())
        n_target_in_subgroup = int(y_sub.sum())
        coverage = n_target_in_subgroup / n_target_total if n_target_total > 0 else 0.0
        lift = target_rate / pop_rate if pop_rate > 0 else np.nan
        support_ratio = n_subgroup / len(y_arr)
        importance = abs(target_rate - pop_rate) * np.sqrt(max(support_ratio, 1e-12))
        rows.append(
            {
                "rule_text": rule_text,
                "n_subgroup": n_subgroup,
                "target_rate": target_rate,
                "coverage": coverage,
                "lift": lift,
                "importance": importance,
            }
        )

    if not rows:
        return []

    rows = sorted(
        rows,
        key=lambda r: (
            r["importance"],
            r["lift"] if not np.isnan(r["lift"]) else -np.inf,
            r["n_subgroup"],
        ),
        reverse=True,
    )
    return rows[:max_rules]


def summarize_rulefit_results(
    rulefit_results: List[Dict[str, Any]],
    target_label: str,
) -> pd.DataFrame:
    """RuleFit 결과를 표준 컬럼 형태로 변환."""
    out_rows: List[Dict[str, Any]] = []
    for i, row in enumerate(rulefit_results, start=1):
        out_rows.append(
            {
                "target_label": target_label,
                "rule_id": i,
                "rule_text": row["rule_text"],
                "n_subgroup": int(row["n_subgroup"]),
                "target_rate": round(float(row["target_rate"]), 4),
                "coverage": round(float(row["coverage"]), 4),
                "lift": round(float(row["lift"]), 4) if not np.isnan(row["lift"]) else np.nan,
                "importance": round(float(row["importance"]), 6),
            }
        )
    return pd.DataFrame(out_rows)


def _extract_rule_variables(rule_text: str) -> List[str]:
    """규칙 문자열에서 변수명만 추출 (비교연산자 + in 구문 지원)."""
    return re.findall(
        r"([A-Za-z_\u3131-\u318E\uAC00-\uD7A3][A-Za-z0-9_\u3131-\u318E\uAC00-\uD7A3]*)\s*(?:!=|<=|>=|<|>|==|\bin\b)",
        str(rule_text),
    )


def summarize_core_variable_axes(
    prim_df: pd.DataFrame,
    rulefit_df: pd.DataFrame,
    target_label: str,
) -> pd.DataFrame:
    """PRIM + RuleFit에서 반복 출현하는 변수 축 요약."""
    prim_counter: Dict[str, int] = {}
    rulefit_counter: Dict[str, int] = {}

    if prim_df is not None and not prim_df.empty:
        for text in prim_df["rule_text"].fillna("").tolist():
            for v in _extract_rule_variables(text):
                prim_counter[v] = prim_counter.get(v, 0) + 1

    if rulefit_df is not None and not rulefit_df.empty:
        for text in rulefit_df["rule_text"].fillna("").tolist():
            for v in _extract_rule_variables(text):
                rulefit_counter[v] = rulefit_counter.get(v, 0) + 1

    all_vars = sorted(set(prim_counter.keys()) | set(rulefit_counter.keys()))
    rows: List[Dict[str, Any]] = []
    for v in all_vars:
        rows.append(
            {
                "target_label": target_label,
                "variable": v,
                "prim_count": prim_counter.get(v, 0),
                "rulefit_count": rulefit_counter.get(v, 0),
                "total_count": prim_counter.get(v, 0) + rulefit_counter.get(v, 0),
            }
        )
    if not rows:
        return pd.DataFrame(columns=["target_label", "variable", "prim_count", "rulefit_count", "total_count"])
    return pd.DataFrame(rows).sort_values(["total_count", "rulefit_count", "prim_count"], ascending=False).reset_index(drop=True)




## 10. 메인 분석 함수 (`run_q11_score_distribution_analysis`)

- Step 1 (PRIM + 5-Fold CV): Q11 1~3점 vs 4~5점, 탈출 방지
- Step 2 (RuleFit + 5-Fold CV): Q11 4점 vs 5점, 팬심 공략

In [ ]:
def run_q11_score_distribution_analysis(config: Config) -> Dict[str, Any]:
    """Q11 기반 Group A/B에 대해 PRIM + RuleFit 분석."""

    global _logger
    _logger = Logger(verbose=config.verbose)

    # ── Random Seed 고정 (재현성 보장) ──
    np.random.seed(config.random_seed)
    import random
    random.seed(config.random_seed)
    log(f"[Random Seed] 고정됨: {config.random_seed} (재현성 보장)")

    base = config.output_dir
    tables_dir = base / "tables"
    results_dir = base / "results"
    ensure_dirs(tables_dir, results_dir)

    log("=" * 80)
    log("Main Analysis: Step1=PRIM(1~3 vs 4~5, 탈출방지+5-Fold CV) / Step2=RuleFit(4 vs 5, 팬심공략)")
    log("=" * 80)

    # ── 데이터 로드 ──
    log(f"[Load] {config.data_csv}")
    df = None
    for enc in ["utf-8", "cp949", "euc-kr"]:
        try:
            df = pd.read_csv(config.data_csv, encoding=enc, low_memory=False)
            log(f"[Load] Success ({enc}): {df.shape}")
            break
        except (UnicodeDecodeError, UnicodeError):
            continue
    if df is None:
        raise ValueError(f"Cannot read file: {config.data_csv}")
    if config.target not in df.columns:
        raise ValueError(f"Target column '{config.target}' not found")

    df = df[df[config.target].notna()].copy()
    q11 = df[config.target].astype(float)

    # 타깃 누수 가능 변수 제외(단, Q12 계열은 설명 변수로 유지)
    base_exclude_cols = {config.target} | {
        c for c in df.columns if any(p in c.lower() for p in ["q13", "q14", "weight"])
    }
    leakage_name_patterns = [
        "target_", "label", "outcome", "pred_", "prediction",
        "non_top", "deepdissatisfaction", "topsatisfaction",
        "y_", "_y", "response_flag",
    ]
    derived_leakage_cols = {
        c for c in df.columns
        if c.lower() != config.target.lower()
        and any(pat in c.lower() for pat in leakage_name_patterns)
    }
    exclude_cols = base_exclude_cols | derived_leakage_cols
    q12_included = [c for c in df.columns if c.startswith("Q12") and c not in exclude_cols]
    log(
        f"[FeatureFilter] excluded={len(exclude_cols)} (Q11/Q13/Q14/weight + derived leakage), "
        f"Q12 included={len(q12_included)}"
    )
    # X_rule: 특수코드 복원·필터 (MICE 미적용)
    X_all, x_rule_report, x_rule_meta = build_x_rule_for_prim_rulefit(
        df,
        exclude_cols,
        config,
        missing_rate_threshold=config.prim_rule_missing_rate_threshold,
    )
    save_csv_helper(
        x_rule_report,
        tables_dir / "X_rule_variable_report.csv",
        log,
        config=config,
        force_save=True,
    )
    ensure_dirs(tables_dir)
    x_rule_path = tables_dir / "X_rule_dataset.csv"
    X_all.to_csv(x_rule_path, index=True, index_label="row_index", encoding="utf-8-sig")
    log(
        f"[Save] {x_rule_path.name} shape={X_all.shape} (index=row_index, PRIM/RuleFit 입력)"
    )
    log(
        f"[X_rule] 결측률 threshold={x_rule_meta['missing_rate_threshold']}, "
        f"후보={x_rule_meta['n_candidate']}, 제거={x_rule_meta['n_removed']}, 최종={x_rule_meta['n_final']}"
    )
    log(f"[X_rule] 제거된 변수 ({len(x_rule_meta['removed_variables'])}): "
        f"{', '.join(x_rule_meta['removed_variables'][:40])}"
        + (" ..." if len(x_rule_meta["removed_variables"]) > 40 else ""))
    log(f"[X_rule] 최종 사용 변수 ({len(x_rule_meta['final_variables'])}): "
        f"{', '.join(x_rule_meta['final_variables'][:40])}"
        + (" ..." if len(x_rule_meta["final_variables"]) > 40 else ""))
    # 코드북 기반 척도 분류 통계 로그
    _cat_cols = set(x_rule_meta.get("categorical_cols", []))
    _num_cols = set(x_rule_meta.get("numeric_cols", []))
    _unk_cols = set(x_rule_meta.get("unknown_cols", []))
    _scale_map = x_rule_meta.get("scale_map", {})
    # PRIM 범주형 peeling: 명목 + (서열 중 명시 구간범주 C* 등). 그 외 서열(리커트 등)은 quantile peeling 유지.
    _explicit_prim_cat = frozenset(config.prim_rule_explicit_categorical_features)
    _prim_cat_cols = {
        c for c, sc in _scale_map.items()
        if sc == "nominal" or (sc == "ordinal" and c in _explicit_prim_cat)
    }
    log(
        f"[Scale] 코드북 척도 분류: "
        f"categorical(명목+서열)={len(_cat_cols)}, "
        f"numeric(비율)={len(_num_cols)}, "
        f"unknown={len(_unk_cols)}, total={X_all.shape[1]}"
    )
    log(
        f"[Feature] X_rule features={X_all.shape[1]} "
        f"(no -69/-77; NaN 허용 → PRIM은 유효값만+명목·지정서열(C*) 범주형 peeling, "
        f"RuleFit은 열별 중앙값 대체)"
    )
    log(
        f"[PRIM] categorical peeling 대상(명목+명시서열)={len(_prim_cat_cols)} "
        f"/ 전체 categorical(명목+서열)={len(_cat_cols)}"
    )
    log(
        f"[PRIM sweep] 고정: min_support=10,20,…,100 (10단위) 별도 CSV "
        f"({len(PRIM_MIN_SUPPORT_SWEEP)}개/그룹, 예: group_a_prim_results_ms30.csv)"
    )

    # ══════════════════════════════════════════════════════════════════════
    # 분석 그룹 정의
    # ──────────────────────────────────────────────────────────────────────
    # Step 1: 1~3점 vs 4~5점  → PRIM   ("탈출 방지": 최악 경험 방지 가이드라인)
    # Step 2: 4점 vs 5점      → RuleFit ("팬심 공략": 보통→충성고객 한 끗 차이)
    # ══════════════════════════════════════════════════════════════════════
    group_defs = [
        {
            "group_key": "step1_escape",
            "target_label": "Q11_1_2_3_vs_4_5",
            "mask": q11.isin([1.0, 2.0, 3.0, 4.0, 5.0]),
            "y_fn": lambda s: (s.isin([1.0, 2.0, 3.0])).astype(int),  # positive=1~3(불만족), negative=4~5(보통이상)
            "pos_desc": "Q11 in {1,2,3} (불만족·탈출 위험)",
            "neg_desc": "Q11 in {4,5} (보통 이상)",
            "algorithm": "prim",
        }
    ]

    prim_cfg = PrimConfig(
        peeling_fraction=0.05,
        min_support=config.prim_min_support,  # 절대 표본수 기준
        min_support_ratio=0.03,   # ← N의 3% 하한 활성화
        max_boxes=5,              # ← 10 → 5
        min_improvement=0.01,     # ← 무의미한 box 방지
        max_peeling_iterations=30,  # ← 50 → 30
    )

    all_group_outputs: Dict[str, Dict[str, pd.DataFrame]] = {}
    merged_rows: List[pd.DataFrame] = []
    axes_rows: List[pd.DataFrame] = []

    def _process_single_group(
        g: Dict[str, Any],
        X_all: pd.DataFrame,
        q11: pd.Series,
        prim_cfg: PrimConfig,
        tables_dir: Path,
        config: Config,
        categorical_cols: Optional[Set[str]] = None,
    ) -> Dict[str, Any]:
        """단일 그룹 분석 (병렬 실행 단위). 즉시 CSV 저장 포함."""
        mask = g["mask"].values
        q11_g = q11.loc[mask]
        X_g = X_all.loc[mask].copy()
        y_g = g["y_fn"](q11_g).values.astype(int)
        n_total = len(y_g)
        n_pos = int((y_g == 1).sum())
        n_neg = int((y_g == 0).sum())
        rate_pos = float(np.mean(y_g)) if n_total > 0 else 0.0
        algo = g["algorithm"]

        log(
            f"\n[{g['group_key'].upper()}] target={g['target_label']} | "
            f"algorithm={algo.upper()} | "
            f"total={n_total}, positive={n_pos}({rate_pos:.4f}), negative={n_neg}"
        )
        log(f"[{g['group_key'].upper()}] positive={g['pos_desc']} / negative={g['neg_desc']}")

        prim_df = pd.DataFrame(
            columns=["target_label", "subgroup_id", "rule_id", "rule_text", "n_subgroup", "target_rate", "coverage", "lift", "method"]
        )
        rulefit_df = pd.DataFrame(
            columns=["target_label", "subgroup_id", "rule_id", "rule_text", "n_subgroup", "target_rate", "coverage", "lift", "importance", "method"]
        )

        if algo == "prim":
            # ── Group A: PRIM only (범주형 변수는 카테고리 peeling 적용) ──
            prim_raw = run_prim_for_target(
                X_g, y_g, prim_cfg, feature_subset=None,
                categorical_cols=categorical_cols,
            )
            prim_df = summarize_prim_results(prim_raw, g["target_label"])
            if not prim_df.empty:
                prim_df["method"] = "PRIM"
                prim_df["rule_id"] = prim_df["subgroup_id"]
                # 모델명 prefix: PRIM_
                prim_df["rule_text"] = "PRIM_" + prim_df["rule_text"].astype(str)
                prim_df = prim_df[
                    ["target_label", "subgroup_id", "rule_id", "rule_text", "n_subgroup", "target_rate", "coverage", "lift", "method"]
                ]

            # ── 5-Fold 교차 검증: Train 룰 → Test 적용 검증 ──
            cv_df = cross_validate_prim(
                X_g, y_g, prim_cfg,
                n_folds=5,
                random_state=config.random_seed,
                target_label=g["target_label"],
                categorical_cols=categorical_cols,
                log_fn=log,
            )
            if not cv_df.empty:
                save_csv_helper(
                    cv_df,
                    tables_dir / f"{g['group_key']}_prim_cv_results.csv",
                    log, config=config, force_save=True,
                )

            # 최소지지도 10~100(10단위) 스윕 → CSV 저장
            save_prim_min_support_sweep_csvs(
                X_g,
                y_g,
                group_key=g["group_key"],
                target_label=g["target_label"],
                tables_dir=tables_dir,
                config=config,
                categorical_cols=categorical_cols,
            )

            # 비동기 저장: PRIM 완료 즉시 개별 CSV
            save_csv_helper(
                prim_df,
                tables_dir / f"{g['group_key']}_prim_results.csv",
                log, config=config, force_save=True,
            )

            if not prim_df.empty:
                log(f"[{g['group_key'].upper()}][PRIM] Top segments:")
                for _, r in prim_df.head(5).iterrows():
                    log(
                        f"  - subgroup_id={int(r['subgroup_id'])}, n={int(r['n_subgroup'])}, "
                        f"rate={r['target_rate']:.4f}, coverage={r['coverage']:.4f}, lift={r['lift']:.4f}, "
                        f"rule={r['rule_text']}"
                    )
            else:
                log(f"[{g['group_key'].upper()}][PRIM] no valid subgroup", level="WARN")

        # 그룹 결과 통합 (해당 알고리즘만 포함)
        merged_df = pd.concat(
            [
                prim_df[["target_label", "subgroup_id", "rule_id", "rule_text", "n_subgroup", "target_rate", "coverage", "lift", "method"]] if not prim_df.empty else pd.DataFrame()
            ],
            ignore_index=True,
        )
        # 비동기 저장: 그룹 결과 즉시 CSV
        save_csv_helper(
            merged_df,
            tables_dir / f"{g['group_key']}_results.csv",
            log, config=config, force_save=True,
        )

        # 변수축 요약 (적용된 알고리즘 기준)
        axes_df = summarize_core_variable_axes(
            prim_df if algo == "prim" else None,
            rulefit_df if algo == "rulefit" else None,
            g["target_label"],
        )
        save_csv_helper(
            axes_df,
            tables_dir / f"{g['group_key']}_core_variable_axes.csv",
            log, config=config, force_save=True,
        )
        if not axes_df.empty:
            top_vars = axes_df.head(10)["variable"].tolist()
            log(f"[{g['group_key'].upper()}] 핵심 변수 축 Top: {', '.join(top_vars)}")

        return {
            "group_key": g["group_key"],
            "prim": prim_df,
            "rulefit": rulefit_df,
            "combined": merged_df,
            "axes": axes_df,
        }

    # ── Step 1(PRIM) 병렬 실행 (ThreadPoolExecutor) ──
    from concurrent.futures import ThreadPoolExecutor
    log("[Parallel] Step1(PRIM) 병렬 실행 시작")

    with ThreadPoolExecutor(max_workers=len(group_defs)) as executor:
        futures = {
            executor.submit(
                _process_single_group, g, X_all, q11, prim_cfg, tables_dir, config,
                _prim_cat_cols,
            ): g["group_key"]
            for g in group_defs
        }
        for future in as_completed(futures):
            gk = futures[future]
            try:
                result = future.result()
                all_group_outputs[result["group_key"]] = {
                    "prim": result["prim"],
                    "rulefit": result["rulefit"],
                    "combined": result["combined"],
                    "axes": result["axes"],
                }
                merged_rows.append(result["combined"])
                axes_rows.append(result["axes"])
                log(f"[Parallel] {gk.upper()} 완료")
            except Exception as e:
                log(f"[Parallel] {gk.upper()} 실패: {e}", level="ERROR")

    # ── RuleFit 관련 코드 제거로 B vs C 비교 비활성화 ──
    b_vs_c_profile_df = pd.DataFrame()

    combined_all = pd.concat(merged_rows, ignore_index=True) if merged_rows else pd.DataFrame()
    axes_all = pd.concat(axes_rows, ignore_index=True) if axes_rows else pd.DataFrame()
    save_csv_helper(combined_all, tables_dir / "all_groups_results_combined.csv", log, config=config, force_save=True)
    save_csv_helper(axes_all, tables_dir / "all_groups_core_variable_axes.csv", log, config=config, force_save=True)

    _logger.save(results_dir / "q11_analysis_log.txt")
    log("\n" + "=" * 80)
    log("Analysis completed: Only PRIM for Q11 1~3 vs 4~5 group was executed.")
    log("=" * 80)

    return {
        "group_results": all_group_outputs,
        "combined_results": combined_all,
        "core_variable_axes": axes_all,
        "group_b_vs_c_profile": b_vs_c_profile_df,
        "X_rule": X_all,
        "X_rule_variable_report": x_rule_report,
        "X_rule_meta": x_rule_meta,
    }

## 11. 진입점 (Entry Points)

`main()`: 기본 output/ 폴더 / `main_for_min_support(n)`: output_ms{n}/ 폴더

In [ ]:

def main():
    """기본 출력 폴더 `output/` (prim_min_support=Config 기본값 10)."""
    config = Config(
        data_csv=_DATA_CSV,
        output_dir=Path("output"),
        target="Q11",
        verbose=True,
    )
    return run_q11_score_distribution_analysis(config)


def main_for_min_support(prim_min_support: int) -> Dict[str, Any]:
    """`output_ms{prim_min_support}/` 에 PRIM·RuleFit·X_rule 전체 저장."""
    config = Config(
        data_csv=_DATA_CSV,
        output_dir=Path(f"output_ms{int(prim_min_support)}"),
        target="Q11",
        verbose=True,
        prim_min_support=int(prim_min_support),
    )
    return run_q11_score_distribution_analysis(config)


# NOTE: `if __name__ == "__main__":` 블록은 main_to_drive를 호출하는 c47fcb84 셀로 이동했습니다.

In [ ]:
def main_to_drive():
    """구글 드라이브 내 출력 폴더 `output/` (prim_min_support=Config 기본값 10)."""
    # Google Drive에 저장할 경로 지정
    drive_output_path = Path("/content/drive/MyDrive/STRICT_ML_Output")
    drive_output_path.mkdir(parents=True, exist_ok=True)

    config = Config(
        data_csv=_DATA_CSV,
        output_dir=drive_output_path,
        target="Q11",
        verbose=True,
    )
    return run_q11_score_distribution_analysis(config)

# 기존 main() 호출 대신 main_to_drive() 호출로 변경
if __name__ == "__main__":
    print("=" * 80)
    print("STRICT_ML: 2단계 Q11 만족도 분석 시작 (Google Drive 저장)")
    print("  Step 1 (PRIM):    1~3점 vs 4~5점 — 탈출 방지 (+ 5-Fold CV)")
    print("  Step 2 (RuleFit): 4점 vs 5점    — 팬심 공략")
    print("=" * 80, flush=True)

    main_to_drive()

    print("\n[Done] 전체 분석 완료.", flush=True)

STRICT_ML: 2단계 Q11 만족도 분석 시작 (Google Drive 저장)
  Step 1 (PRIM):    1~3점 vs 4~5점 — 탈출 방지 (+ 5-Fold CV)
  Step 2 (RuleFit): 4점 vs 5점    — 팬심 공략
[07:44:31] [INFO] [Random Seed] 고정됨: 2026 (재현성 보장)
[07:44:31] [INFO] ================================================================================
[07:44:31] [INFO] Main Analysis: Step1=PRIM(1~3 vs 4~5, 탈출방지+5-Fold CV) / Step2=RuleFit(4 vs 5, 팬심공략)
[07:44:31] [INFO] ================================================================================
[07:44:31] [INFO] [Load] 2024_외래관광객조사_Data_수정_no_col1.csv
[07:44:32] [INFO] [Load] Success (cp949): (16216, 401)
[07:44:33] [INFO] [FeatureFilter] excluded=8 (Q11/Q13/Q14/weight + derived leakage), Q12 included=27
[07:44:35] [INFO] [Save] X_rule_variable_report.csv (393 rows)
[07:44:36] [INFO] [Save] X_rule_dataset.csv shape=(16216, 137) (index=row_index, PRIM/RuleFit 입력)
[07:44:36] [INFO] [X_rule] 결측률 threshold=0.5, 후보=393, 제거=256, 최종=137
[07:44:36] [INFO] [X_rule] 제거된 변수 (256): XRVIT, Q1_1a3, Q2a2, Q

## 12. 실행

아래 셀에서 분석을 실행합니다. `data_csv` 경로를 실제 데이터 파일 위치로 수정하세요.

In [ ]:
# 기본 분석 실행 (output/ 폴더)
if __name__ == '__main__' or True:
    results = main()
    print('분석 완료')


[07:46:13] [INFO] [Random Seed] 고정됨: 2026 (재현성 보장)
[07:46:13] [INFO] ================================================================================
[07:46:13] [INFO] Main Analysis: Step1=PRIM(1~3 vs 4~5, 탈출방지+5-Fold CV) / Step2=RuleFit(4 vs 5, 팬심공략)
[07:46:13] [INFO] ================================================================================
[07:46:13] [INFO] [Load] 2024_외래관광객조사_Data_수정_no_col1.csv
[07:46:14] [INFO] [Load] Success (cp949): (16216, 401)
[07:46:14] [INFO] [FeatureFilter] excluded=8 (Q11/Q13/Q14/weight + derived leakage), Q12 included=27
[07:46:15] [INFO] [Save] X_rule_variable_report.csv (393 rows)
[07:46:16] [INFO] [Save] X_rule_dataset.csv shape=(16216, 137) (index=row_index, PRIM/RuleFit 입력)
[07:46:16] [INFO] [X_rule] 결측률 threshold=0.5, 후보=393, 제거=256, 최종=137
[07:46:16] [INFO] [X_rule] 제거된 변수 (256): XRVIT, Q1_1a3, Q2a2, Q2a3, Q2a_dk, Q2_1a2, Q2_1a3, ZQ2_31, ZQ2_32, ZQ2_33, ZQ2_34, Q2_3a2at, Q2_3a2bt, Q2_3a2ct, Q2_3a2dt, Q2_3a2et, Q2_3a3at, Q2_3a3bt, Q2_3a3ct, Q

## 13. CHAID-like 분석 (Decision Tree)

In [9]:
from __future__ import annotations

import os
import sys

import numpy as np
import pandas as pd
from pathlib import Path # Import Path for DATA_PATH

try:
    from CHAID import Tree
except ImportError as exc:  # pragma: no cover
    sys.stderr.write(
        "The 'CHAID' package is required. Install it with `pip install CHAID`.\n"
    )
    raise

DATA_PATH = Path("2024_외래관광객조사_Data_수정_no_col1.csv") # Corrected to the actual data file path
OUTPUT_DIR = "output/chaid"

PREDICTORS = {
    "D_MON": "월",
    "D_BUN": "분기",
    "D_NAT": "국적",
    "D_SEX": "성별",
    "D_AGE": "연령별",
    "D_MOK": "방한 목적별",
    "D_NUM": "방한 횟수별",
    "D_GUB": "여행 형태별",
}

TARGET_COL = "TARGET"


def build_target(df: pd.DataFrame) -> pd.Series:
    """Return 1 if (Q12a19 < 5) AND (Q12a19 < 4) AND (Q12a02 < 5), else 0.

    Rows where any of the source variables is missing become NaN and are
    dropped prior to fitting.
    """
    q12a19 = pd.to_numeric(df["Q12a19"], errors="coerce")
    q12a02 = pd.to_numeric(df["Q12a02"], errors="coerce")
    cond = (q12a19 < 5) & (q12a19 < 4) & (q12a02 < 5)
    target = cond.astype("float")
    target[q12a19.isna() | q12a02.isna()] = np.nan
    return target


def main() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(DATA_PATH, low_memory=False)

    missing = [c for c in list(PREDICTORS) + ["Q12a19", "Q12a02"] if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in data: {missing}")

    df[TARGET_COL] = build_target(df)

    model_df = df[list(PREDICTORS) + [TARGET_COL]].dropna().copy()
    for col in PREDICTORS:
        model_df[col] = model_df[col].astype("category")
    model_df[TARGET_COL] = model_df[TARGET_COL].astype(int)

    n_total = len(model_df)
    n_pos = int(model_df[TARGET_COL].sum())
    print(f"[INFO] rows used: {n_total}")
    print(f"[INFO] positive target (=1): {n_pos} ({n_pos / n_total:.3%})")

    # The CHAID package applies a Bonferroni correction to the split p-values
    # when split_threshold=0 and is_exhaustive=False (classic CHAID with
    # Bonferroni adjustment as described by Kass, 1980).
    independent_variable_columns = list(PREDICTORS)
    dep_variable = TARGET_COL

    tree = Tree.from_pandas_df(
        model_df,
        i_variables={col: "nominal" for col in independent_variable_columns},
        d_variable=dep_variable,
        dep_variable_type="categorical",
        alpha_merge=0.05,
        max_depth=5,
        min_parent_node_size=30,
        min_child_node_size=15,
        split_threshold=0,          # enable Bonferroni-corrected p-values
        is_exhaustive=False,        # classic CHAID (not exhaustive CHAID)
    )

    # --- text report -------------------------------------------------------
    report_path = os.path.join(OUTPUT_DIR, "chaid_tree.txt")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("CHAID (Bonferroni-adjusted) tree\n")
        f.write("=" * 60 + "\n")
        f.write(
            "Target: (Q12a19 < 5) AND (Q12a19 < 4) AND (Q12a02 < 5)\n"
        )
        f.write(f"Rows used: {n_total}\n")
        f.write(f"Positive rate: {n_pos / n_total:.4f}\n\n")
        f.write("Predictors: " + ", ".join(
            f"{k} ({v})" for k, v in PREDICTORS.items()
        ) + "\n\n")
        f.write(str(tree) + "\n")
    print(f"[INFO] tree report saved to {report_path}")

    # --- node table --------------------------------------------------------
    rows = []
    for node in tree:
        members = node.members if isinstance(node.members, dict) else {}
        rows.append(
            {
                "node_id": node.node_id,
                "parent": node.parent,
                "depth": node.depth,
                "is_terminal": node.is_terminal,
                "split_variable": node.split_variable,
                "choices": str(node.choices) if node.choices is not None else "",
                "p": getattr(node, "p", None),
                "score": getattr(node, "score", None),
                "n": sum(members.values()) if members else 0,
                "n_pos": int(members.get(1, 0)) if members else 0,
                "n_neg": int(members.get(0, 0)) if members else 0,
            }
        )
    nodes_df = pd.DataFrame(rows)
    nodes_csv = os.path.join(OUTPUT_DIR, "chaid_nodes.csv")
    nodes_df.to_csv(nodes_csv, index=False, encoding="utf-8-sig")
    print(f"[INFO] node table saved to {nodes_csv}")

    # --- graphviz (optional) ----------------------------------------------
    try:
        dot_path = os.path.join(OUTPUT_DIR, "chaid_tree")
        tree.render(path=dot_path, view=False)
        print(f"[INFO] graph rendered to {dot_path}.gv/.pdf")
    except Exception as exc:  # rendering requires graphviz
        print(f"[WARN] tree.render skipped: {exc}")


if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: '2024_외래관광객조사_Data_수정_no_col1.csv'

In [3]:
# Install the CHAID package
!pip install CHAID

In [27]:
import os
import sys

import numpy as np
import pandas as pd
from pathlib import Path # Import Path for DATA_PATH

try:
    from CHAID import Tree
except ImportError as exc:  # pragma: no cover
    sys.stderr.write(
        "The 'CHAID' package is required. Install it with `pip install CHAID`.\n"
    )
    raise

# Use the globally defined _DATA_CSV for consistency (now an absolute Path object)
DATA_PATH = _DATA_CSV
# OUTPUT_DIR will now be passed as an argument to run_chaid_analysis

PREDICTORS = {
    "D_MON": "월",
    "D_BUN": "분기",
    "D_NAT": "국적",
    "D_SEX": "성별",
    "D_AGE": "연령별",
    "D_MOK": "방한 목적별",
    "D_NUM": "방한 횟수별",
    "D_GUB": "여행 형태별",
}

TARGET_COL = "TARGET"

def build_target(df: pd.DataFrame) -> pd.Series:
    """Return 0 if (Q12a19 < 4) AND (Q12a02 < 5), else 1.

    Rows where any of the source variables is missing become NaN and are
    dropped prior to fitting.
    """
    q12a19 = pd.to_numeric(df["Q12a19"], errors="coerce")
    q12a02 = pd.to_numeric(df["Q12a02"], errors="coerce")
    # Condition: (Q12a19 < 4) & (Q12a02 < 5) indicates dissatisfaction condition.
    # Per user's clarification: Target 0 = Dissatisfaction, Target 1 = Not Dissatisfaction.
    # So, if cond is True, target should be 0.
    cond = (q12a19 < 4) & (q12a02 < 5)
    target = pd.Series(np.nan, index=df.index, dtype=float)
    target[cond] = 0.0 # Dissatisfied
    target[~cond] = 1.0 # Not Dissatisfied

    target[q12a19.isna() | q12a02.isna()] = np.nan
    return target


def run_chaid_analysis(output_dir: str = "output/chaid"):
    os.makedirs(output_dir, exist_ok=True)

    # DATA_PATH is now an absolute Path object from _DATA_CSV, so no need for os.path.abspath()
    print(f"[INFO] Attempting to read data from: {DATA_PATH}")
    df = pd.read_csv(DATA_PATH, low_memory=False, encoding='cp949') # Added encoding='cp949'

    missing = [c for c in list(PREDICTORS) + ["Q12a19", "Q12a02"] if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in data: {missing}")

    df[TARGET_COL] = build_target(df)

    model_df = df[list(PREDICTORS) + [TARGET_COL]].dropna().copy()
    for col in PREDICTORS:
        model_df[col] = model_df[col].astype("category")
    model_df[TARGET_COL] = model_df[TARGET_COL].astype(int)

    n_total = len(model_df)
    n_dissatisfied = int((model_df[TARGET_COL] == 0).sum()) # Count of 0s (dissatisfied)
    n_not_dissatisfied = int((model_df[TARGET_COL] == 1).sum()) # Count of 1s (not dissatisfied)

    # Calculate overall dissatisfaction rate for Lift calculation
    overall_dissatisfaction_rate = n_dissatisfied / n_total if n_total > 0 else 0

    print(f"[INFO] rows used: {n_total}")
    print(f"[INFO] target=0 (dissatisfaction): {n_dissatisfied} ({n_dissatisfied / n_total:.3%})")
    print(f"[INFO] target=1 (non-dissatisfaction): {n_not_dissatisfied} ({n_not_dissatisfied / n_total:.3%})")

    independent_variable_columns = list(PREDICTORS)
    dep_variable = TARGET_COL

    tree = Tree.from_pandas_df(
        model_df,
        i_variables={col: "nominal" for col in independent_variable_columns},
        d_variable=dep_variable,
        dep_variable_type="categorical",
        alpha_merge=0.05, # User specified alpha_merge=0.05
        max_depth=5,
        min_parent_node_size=30,
        min_child_node_size=15,
        split_threshold=0,          # enable Bonferroni-corrected p-values
        is_exhaustive=False,        # classic CHAID (not exhaustive CHAID)
    )

    # --- text report -------------------------------------------------------
    report_path = os.path.join(output_dir, "chaid_tree.txt")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("CHAID (Bonferroni-adjusted) tree\n")
        f.write("=" * 60 + "\n")
        f.write(
            "Target: 0 if (Q12a19 < 4) AND (Q12a02 < 5) (Dissatisfaction), else 1 (Not Dissatisfaction)\n"
        )
        f.write(f"Rows used: {n_total}\n")
        f.write(f"Target=0 (Dissatisfaction) rate: {n_dissatisfied / n_total:.4f}\n\n")
        f.write("Predictors: " + ", ".join(
            f"{k} ({v})" for k, v in PREDICTORS.items()
        ) + "\n\n")
        f.write(str(tree) + "\n")
    print(f"[INFO] tree report saved to {report_path}")

    # --- Pre-calculate children for each node to get 'k' for Cramer's V ---
    children_map = {}
    for n_node in tree:
        if n_node.parent is not None:
            if n_node.parent not in children_map:
                children_map[n_node.parent] = []
            children_map[n_node.parent].append(n_node.node_id)

    # --- node table --------------------------------------------------------
    rows = []
    for node in tree:
        members = node.members if isinstance(node.members, dict) else {}
        node_n = sum(members.values()) if members else 0
        node_n_dissatisfied = int(members.get(0, 0)) if members else 0
        node_dissatisfaction_rate = node_n_dissatisfied / node_n if node_n > 0 else 0.0
        node_lift = node_dissatisfaction_rate / overall_dissatisfaction_rate if overall_dissatisfaction_rate > 0 else np.nan

        cramers_v = np.nan
        # Calculate Cramer's V only for splitting nodes
        if node.split_variable is not None and not node.is_terminal:
            chi_square_val = getattr(node, "score", None)
            if chi_square_val is not None and node_n > 0:
                # k is the number of categories/groups the split variable divides into
                # which is equivalent to the number of immediate children of this splitting node.
                k = len(children_map.get(node.node_id, [])) # Get count of children for this node
                r = 2 # Number of categories in target variable (0 and 1)

                denominator = min(k - 1, r - 1)

                if denominator > 0:
                    cramers_v = np.sqrt((chi_square_val / node_n) / denominator)
                else:
                    cramers_v = 0.0 # Should not happen for a valid split, but for safety

        rows.append(
            {
                "node_id": node.node_id,
                "parent": node.parent,
                "depth": getattr(node, "depth", None), # Safely access depth attribute
                "is_terminal": node.is_terminal,
                "split_variable": node.split_variable,
                "choices": str(node.choices) if node.choices is not None else "",
                "p": getattr(node, "p", None),
                "score": getattr(node, "score", None),
                "n": node_n,
                "n_pos": node_n_dissatisfied, # Count of Target 0 (Dissatisfaction)
                "n_neg": int(members.get(1, 0)) if members else 0, # Count of Target 1 (Not Dissatisfaction)
                "rate": node_dissatisfaction_rate,
                "lift": node_lift,
                "cramers_v": cramers_v,
            }
        )
    nodes_df = pd.DataFrame(rows)
    nodes_csv = os.path.join(output_dir, "chaid_nodes.csv")
    nodes_df.to_csv(nodes_csv, index=False, encoding="utf-8-sig")
    print(f"[INFO] node table saved to {nodes_csv}")

    # --- graphviz (optional) ----------------------------------------------
    try:
        dot_path = os.path.join(output_dir, "chaid_tree")
        tree.render(path=dot_path, view=False) # Render to file without displaying
        print(f"[INFO] graph rendered to {dot_path}.gv/.pdf")
    except Exception as exc:  # rendering requires graphviz
        print(f"[WARN] tree.render skipped: {exc}")

    # --- Worst Case Target Paths (Highest Dissatisfaction Rate) ---
    print("\n[Worst Case Target Paths (Highest Dissatisfaction Rate)]:")
    worst_paths_info = []

    # Create a map of node_id to Node object for easy lookup
    nodes_by_id = {node.node_id: node for node in tree}

    # Find leaf nodes and their dissatisfaction rates
    leaf_nodes = [node for node in tree if node.is_terminal]

    if not leaf_nodes:
        print("No terminal (leaf) nodes found in the tree.")
    else:
        max_dissatisfaction_rate = -1.0
        # Target '0' is dissatisfaction (as per new build_target function)
        # So, we are looking for the highest rate of '0's
        for leaf_node in leaf_nodes:
            members = leaf_node.members # target_distribution is members
            total_samples = sum(members.values())
            if total_samples == 0:
                continue

            dissatisfaction_rate = members.get(0, 0) / total_samples # Get rate of target 0
            node_lift = dissatisfaction_rate / overall_dissatisfaction_rate if overall_dissatisfaction_rate > 0 else np.nan

            if dissatisfaction_rate > max_dissatisfaction_rate:
                max_dissatisfaction_rate = dissatisfaction_rate
                worst_paths_info = [(leaf_node, dissatisfaction_rate, node_lift, total_samples)]
            elif dissatisfaction_rate == max_dissatisfaction_rate:
                worst_paths_info.append((leaf_node, dissatisfaction_rate, node_lift, total_samples))

        if worst_paths_info:
            for leaf_node, rate, lift, n_samples in worst_paths_info:
                path_rules = []
                current_node = leaf_node

                # Traverse up from leaf to root
                while current_node.parent is not None: # Root has parent as None
                    parent_node_id = current_node.parent
                    parent_node = nodes_by_id[parent_node_id]

                    parent_split_var = getattr(parent_node, 'split_variable', None)
                    choice_values = current_node.choices

                    if parent_split_var is None or choice_values is None:
                        path_rules.insert(0, f"Unknown split for node {current_node.node_id} (parent_split_var or choices missing)")
                        break # Cannot reconstruct path if critical info is missing

                    # Format rule using parent_split_var and choice_values
                    if isinstance(choice_values, (tuple, list)):
                        path_rules.insert(0, f"{parent_split_var} in {{{', '.join(map(str, choice_values))}}}")
                    else:
                        path_rules.insert(0, f"{parent_split_var} == {choice_values}")

                    current_node = parent_node # Move up to the parent

                print(f"  - 불만족 비율: {rate:.4f} (표본 수: {n_samples}), 리프트: {lift:.4f}")
                print(f"    경로: {' AND '.join(path_rules)}")
        else:
            print("불만족 비율이 높은 리프 노드를 찾을 수 없습니다.")


In [28]:
from google.colab import drive

def run_chaid_to_drive():
    """Run CHAID analysis and save results to Google Drive."""
    drive_output_path = Path("/content/drive/MyDrive/STRICT_ML_Output/chaid")
    drive_output_path.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Saving CHAID results to Google Drive at: {drive_output_path}")
    run_chaid_analysis(output_dir=str(drive_output_path))

if __name__ == "__main__":
    run_chaid_to_drive()

[INFO] Saving CHAID results to Google Drive at: /content/drive/MyDrive/STRICT_ML_Output/chaid
[INFO] Attempting to read data from: /content/PORT_ONE/2024_외래관광객조사_Data_수정_no_col1.csv
[INFO] rows used: 12928
[INFO] target=0 (dissatisfaction): 360 (2.785%)
[INFO] target=1 (non-dissatisfaction): 12568 (97.215%)
[INFO] tree report saved to /content/drive/MyDrive/STRICT_ML_Output/chaid/chaid_tree.txt
[INFO] node table saved to /content/drive/MyDrive/STRICT_ML_Output/chaid/chaid_nodes.csv
[WARN] tree.render skipped: 
The orca executable is required to export figures as static images,
but it could not be found on the system path.

Searched for executable 'orca' on the following path:
    /opt/bin
    /usr/local/sbin
    /usr/local/bin
    /usr/sbin
    /usr/bin
    /sbin
    /bin
    /tools/node/bin
    /tools/google-cloud-sdk/bin

If you haven't installed orca yet, you can do so using conda as follows:

    $ conda install -c plotly plotly-orca

Alternatively, see other installation methods i

In [29]:
import pandas as pd
from pathlib import Path
import os

drive_output_path = Path("/content/drive/MyDrive/STRICT_ML_Output/chaid")
nodes_csv_path = drive_output_path / "chaid_nodes.csv"

if nodes_csv_path.exists():
    nodes_df = pd.read_csv(nodes_csv_path, encoding='utf-8-sig')
    print(f"[{nodes_csv_path.name}] 파일의 상위 10개 행:")
    display(nodes_df.head(10))
else:
    print(f"오류: {nodes_csv_path} 파일을 찾을 수 없습니다. CHAID 분석이 올바르게 실행되었는지 확인해주세요.")

[chaid_nodes.csv] 파일의 상위 10개 행:


,node_id,parent,depth,is_terminal,split_variable,choices,p,score,n,n_pos,n_neg,rate,lift,cramers_v
0,0,NaN,NaN,False,D_NAT,[],2.135743e-27,131.219670,12928.0,360,12568,0.027847,1.000000,0.100747
1,1,0.0,NaN,False,D_MON,"[1, 4, 20, 14, 17, 97]",4.472004e-03,8.081475,4039.0,78,3961,0.019312,0.693505,0.044731
2,2,1.0,NaN,False,D_GUB,"[1, 6, 10, 5, 7, 12]",2.012865e-02,5.400702,2049.0,52,1997,0.025378,0.911361,0.051340
3,3,2.0,NaN,False,D_NAT,"[1, 2]",2.280034e-02,5.183653,1902.0,44,1858,0.023134,0.830751,0.052205
4,4,3.0,NaN,False,D_AGE,"[1, 4, 20]",7.001505e-03,7.272583,1295.0,23,1272,0.017761,0.637804,0.074939
5,5,4.0,NaN,True,NaN,"[1, 2]",NaN,NaN,494.0,15,479,0.030364,1.090418,NaN
6,6,4.0,NaN,True,NaN,"[3, 4, 5, 6]",NaN,NaN,801.0,8,793,0.009988,0.358663,NaN
7,7,3.0,NaN,False,D_MON,"[14, 97, 17]",4.417062e-04,12.346956,607.0,21,586,0.034596,1.242394,0.142622
8,8,7.0,NaN,True,NaN,"[1, 5, 6, 10]",NaN,NaN,415.0,7,408,0.016867,0.605730,NaN
9,9,7.0,NaN,True,NaN,"[7, 12]",NaN,NaN,192.0,14,178,0.072917,2.618519,NaN


In [30]:
chaid_summary_path = Path("/content/chaid_final_summary.csv")

if chaid_summary_path.exists():
    chaid_summary_df = pd.read_csv(chaid_summary_path, encoding='utf-8-sig')
    print(f"[{chaid_summary_path.name}] 파일의 상위 5개 행:")
    display(chaid_summary_df.head())
else:
    print(f"오류: {chaid_summary_path} 파일을 찾을 수 없습니다.")


[chaid_final_summary.csv] 파일의 상위 5개 행:


,split_variable_parent,choices,n,n_pos,node_id
0,D_AGE,"[1, 2]",494,15,5
1,D_AGE,"[3, 4, 5, 6]",801,8,6
2,D_AGE,[1],33,2,18
3,D_AGE,"[1, 2, 3, 4]",37,0,32
4,D_AGE,"[5, 6]",16,3,33


In [31]:
import os

# 'output' 디렉토리 및 모든 하위 디렉토리의 파일 목록을 재귀적으로 가져옵니다.
def list_files_in_directory(directory_path):
    file_list = []
    for root, dirs, files in os.walk(directory_path):
        for file in files:
            file_list.append(os.path.join(root, file))
    return file_list

output_dir = 'output'
files = list_files_in_directory(output_dir)

if files:
    print(f"'{output_dir}' 디렉토리 내에 저장된 파일 목록:")
    for f in files:
        print(f)
else:
    print(f"'{output_dir}' 디렉토리가 없거나 파일이 저장되지 않았습니다.")

'output' 디렉토리 내에 저장된 파일 목록:
output/prim_rulefit_report.html
output/results/q11_analysis_log.txt
output/tables/group_b_rulefit_linear_terms.csv
output/tables/all_groups_core_variable_axes.csv
output/tables/group_c_rulefit_linear_terms.csv
output/tables/group_a_core_variable_axes.csv
output/tables/group_b_rulefit_CB_results.csv
output/tables/group_c_rulefit_GBM_results.csv
output/tables/group_b_vs_c_contrasting_profile_summary.csv
output/tables/group_b_results.csv
output/tables/X_rule_dataset.csv
output/tables/group_a_prim_results_ms20.csv
output/tables/group_c_rulefit_LGBM_results.csv
output/tables/group_b_rulefit_LGBM_results.csv
output/tables/group_a_prim_results_ms30.csv
output/tables/group_a_prim_results_ms100.csv
output/tables/group_b_rulefit_results.csv
output/tables/group_a_prim_results_ms10.csv
output/tables/group_c_results.csv
output/tables/group_a_results.csv
output/tables/group_a_prim_results.csv
output/tables/group_c_rulefit_XGB_results.csv
output/tables/group_b_core_variabl

## 13. 로지스틱 회귀 분석

In [32]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

# DATA_PATH는 전역 변수 _DATA_CSV를 사용합니다.
DATA_PATH = _DATA_CSV

PREDICTORS = {
    "D_MON": "월",
    "D_BUN": "분기",
    "D_NAT": "국적",
    "D_SEX": "성별",
    "D_AGE": "연령별",
    "D_MOK": "방한 목적별",
    "D_NUM": "방한 횟수별",
    "D_GUB": "여행 형태별",
}

TARGET_COL = "TARGET"

def build_target(df: pd.DataFrame) -> pd.Series:
    """Return 0 if (Q12a19 < 4) AND (Q12a02 < 5), else 1.

    Rows where any of the source variables is missing become NaN and are
    dropped prior to fitting.
    """
    q12a19 = pd.to_numeric(df["Q12a19"], errors="coerce")
    q12a02 = pd.to_numeric(df["Q12a02"], errors="coerce")
    cond = (q12a19 < 4) & (q12a02 < 5)
    target = pd.Series(np.nan, index=df.index, dtype=float)
    target[cond] = 0.0 # Dissatisfied
    target[~cond] = 1.0 # Not Dissatisfied
    target[q12a19.isna() | q12a02.isna()] = np.nan
    return target

print("필요한 라이브러리 및 함수 정의 완료")

필요한 라이브러리 및 함수 정의 완료


### 13.1 데이터 로드 및 전처리

In [33]:
# 데이터 로드
print(f"[INFO] Attempting to read data from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH, low_memory=False, encoding='cp949')

# 타겟 변수 생성
df[TARGET_COL] = build_target(df)

# 예측 변수와 타겟 변수 선택 및 결측값 제거
predictor_cols = list(PREDICTORS.keys())
model_df = df[predictor_cols + [TARGET_COL]].dropna().copy()

# 범주형 변수 처리 (One-Hot Encoding)
# CHAID는 자체적으로 범주형 변수를 처리하지만, 로지스틱 회귀는 수치형 입력이 필요합니다.

# 예측 변수 (X)와 타겟 변수 (y) 분리
X = model_df[predictor_cols]
y = model_df[TARGET_COL].astype(int)

# 범주형 변수 목록
categorical_features = predictor_cols

# 원-핫 인코더를 포함하는 ColumnTransformer 생성
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # 범주형이 아닌 다른 특성은 그대로 통과
)

print(f"총 {len(model_df)}개의 샘플 사용.")
print(f"타겟 0 (불만족) 비율: {(y == 0).sum() / len(y):.3%}")
print(f"타겟 1 (불만족 아님) 비율: {(y == 1).sum() / len(y):.3%}")

[INFO] Attempting to read data from: /content/PORT_ONE/2024_외래관광객조사_Data_수정_no_col1.csv
총 12928개의 샘플 사용.
타겟 0 (불만족) 비율: 2.785%
타겟 1 (불만족 아님) 비율: 97.215%


### 13.2 모델 학습 및 평가

In [34]:
# 훈련 및 테스트 세트 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 로지스틱 회귀 모델 파이프라인 생성
# Solver는 'liblinear'를 사용하여 작은 데이터셋에 적합하고 L1/L2 정규화 지원
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='liblinear', random_state=42))
])

# 모델 훈련
print("로지스틱 회귀 모델 훈련 시작...")
model.fit(X_train, y_train)
print("로지스틱 회귀 모델 훈련 완료.")

# 예측 및 평가
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1] # 클래스 1 (불만족 아님)에 대한 확률

print("\n--- 모델 평가 보고서 ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

로지스틱 회귀 모델 훈련 시작...
로지스틱 회귀 모델 훈련 완료.

--- 모델 평가 보고서 ---
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       108
           1       0.97      1.00      0.99      3771

    accuracy                           0.97      3879
   macro avg       0.49      0.50      0.49      3879
weighted avg       0.95      0.97      0.96      3879

ROC-AUC Score: 0.6105


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



### 13.3 모델 계수 해석 (선택 사항)

In [35]:
try:
    # 원-핫 인코딩된 특성 이름 가져오기
    ohe_feature_names = model.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features)
    all_feature_names = list(ohe_feature_names)

    # 로지스틱 회귀 계수 (coefficient) 가져오기
    coefficients = model.named_steps['classifier'].coef_[0]

    # 계수와 특성 이름을 데이터프레임으로 결합
    coef_df = pd.DataFrame({'Feature': all_feature_names, 'Coefficient': coefficients})
    coef_df['Absolute_Coefficient'] = np.abs(coef_df['Coefficient'])

    # 절대값 기준으로 정렬하여 중요도 파악
    coef_df = coef_df.sort_values(by='Absolute_Coefficient', ascending=False)

    print("\n--- 로지스틱 회귀 계수 (절대값 기준 정렬) ---")
    display(coef_df.head(10))

    print("\n* 계수 해석:")
    print("  - 양수 계수: 해당 특성이 타겟 클래스 1 (불만족 아님)에 긍정적인 영향을 미침 (즉, 불만족을 낮춤)")
    print("  - 음수 계수: 해당 특성이 타겟 클래스 1 (불만족 아님)에 부정적인 영향을 미침 (즉, 불만족을 높임)")
    print("  - 절대값이 클수록 타겟에 미치는 영향이 큼")

except Exception as e:
    print(f"계수 해석 중 오류 발생: {e}")
    print("원-핫 인코딩된 특성 이름을 가져오거나 계수를 해석하는 데 문제가 있을 수 있습니다.")


--- 로지스틱 회귀 계수 (절대값 기준 정렬) ---


,Feature,Coefficient,Absolute_Coefficient
34,D_NAT_19,1.296011,1.296011
17,D_NAT_2,-0.954794,0.954794
46,D_MOK_2,0.939779,0.939779
21,D_NAT_6,0.938398,0.938398
28,D_NAT_13,0.827362,0.827362
33,D_NAT_18,0.823801,0.823801
37,D_SEX_1,0.797574,0.797574
25,D_NAT_10,-0.786931,0.786931
20,D_NAT_5,-0.721146,0.721146
56,D_GUB_3,0.703701,0.703701



* 계수 해석:
  - 양수 계수: 해당 특성이 타겟 클래스 1 (불만족 아님)에 긍정적인 영향을 미침 (즉, 불만족을 낮춤)
  - 음수 계수: 해당 특성이 타겟 클래스 1 (불만족 아님)에 부정적인 영향을 미침 (즉, 불만족을 높임)
  - 절대값이 클수록 타겟에 미치는 영향이 큼


In [36]:
from pathlib import Path
import os

# Google Drive에 저장할 경로 지정
drive_output_path = Path("/content/drive/MyDrive/STRICT_ML_Output/logistic_regression")
drive_output_path.mkdir(parents=True, exist_ok=True)

# coef_df DataFrame을 CSV로 저장
coef_csv_path = drive_output_path / "logistic_regression_coefficients.csv"
coef_df.to_csv(coef_csv_path, index=False, encoding='utf-8-sig')

print(f"[INFO] 로지스틱 회귀 계수 CSV 파일이 다음 경로에 저장되었습니다: {coef_csv_path}")

[INFO] 로지스틱 회귀 계수 CSV 파일이 다음 경로에 저장되었습니다: /content/drive/MyDrive/STRICT_ML_Output/logistic_regression/logistic_regression_coefficients.csv


In [37]:
# coef_df에 오즈비(Odds Ratio) 열 추가
coef_df['Odds_Ratio'] = np.exp(coef_df['Coefficient'])

print("\n--- 로지스틱 회귀 계수와 오즈비 (절대값 기준 정렬) ---")
display(coef_df.head(10))

print("\n* 오즈비 해석:")
print("  - 오즈비가 1보다 크면: 해당 특성이 타겟 클래스 1 (불만족 아님)이 될 오즈를 증가시킴 (불만족도를 낮춤)")
print("  - 오즈비가 1보다 작으면: 해당 특성이 타겟 클래스 1 (불만족 아님)이 될 오즈를 감소시킴 (불만족도를 높임)")
print("  - 오즈비가 1이면: 해당 특성이 타겟에 영향을 미치지 않음")

# 업데이트된 coef_df를 Google Drive에 다시 저장 (옵션)
drive_output_path = Path("/content/drive/MyDrive/STRICT_ML_Output/logistic_regression")
coef_csv_path_with_odds = drive_output_path / "logistic_regression_coefficients_with_odds.csv"
coef_df.to_csv(coef_csv_path_with_odds, index=False, encoding='utf-8-sig')

print(f"[INFO] 로지스틱 회귀 계수 및 오즈비 CSV 파일이 다음 경로에 저장되었습니다: {coef_csv_path_with_odds}")


--- 로지스틱 회귀 계수와 오즈비 (절대값 기준 정렬) ---


,Feature,Coefficient,Absolute_Coefficient,Odds_Ratio
34,D_NAT_19,1.296011,1.296011,3.654690
17,D_NAT_2,-0.954794,0.954794,0.384891
46,D_MOK_2,0.939779,0.939779,2.559416
21,D_NAT_6,0.938398,0.938398,2.555883
28,D_NAT_13,0.827362,0.827362,2.287277
33,D_NAT_18,0.823801,0.823801,2.279146
37,D_SEX_1,0.797574,0.797574,2.220148
25,D_NAT_10,-0.786931,0.786931,0.455240
20,D_NAT_5,-0.721146,0.721146,0.486195
56,D_GUB_3,0.703701,0.703701,2.021220



* 오즈비 해석:
  - 오즈비가 1보다 크면: 해당 특성이 타겟 클래스 1 (불만족 아님)이 될 오즈를 증가시킴 (불만족도를 낮춤)
  - 오즈비가 1보다 작으면: 해당 특성이 타겟 클래스 1 (불만족 아님)이 될 오즈를 감소시킴 (불만족도를 높임)
  - 오즈비가 1이면: 해당 특성이 타겟에 영향을 미치지 않음
[INFO] 로지스틱 회귀 계수 및 오즈비 CSV 파일이 다음 경로에 저장되었습니다: /content/drive/MyDrive/STRICT_ML_Output/logistic_regression/logistic_regression_coefficients_with_odds.csv
